# CS781 Phase 2 — improve submissions 01, 03 and 05

Run All in the same Kaggle environment as the previous Phase-2 run.  This
notebook keeps the old Phase-2 output directory unchanged and writes new work
under `/kaggle/working/CS781_Capstone/phase2_improved`.

It runs three document-level validation folds per method, then trains final
models from the matching Phase-1 checkpoints, makes resumable official-test
predictions and exports three **new** validated ZIPs.  Every training stage has
its own checkpoint, history and uniquely named recovery ZIP.  If a session
time guard stops work, attach the new recovery ZIPs as Kaggle Inputs and Run All
again.  No notebook can guarantee an uninterrupted Kaggle session, exact GPU
utilization or a higher official score.

The methods and hyperparameters are AI-assisted.  Follow your course's
authorship and disclosure rules.


In [ ]:
from __future__ import annotations

import ast
import hashlib
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Sampler


# Run All executes the six AI-assisted reference experiments when inputs exist.
# This is not a student-authorship attestation or a guarantee of course acceptance.
RUN_PHASE2 = True
STUDENT_AUTHORSHIP_CONFIRMED = False
GPU_SESSION_BUDGET_SECONDS = 31_800
INFERENCE_RECOVERY_INTERVAL = 250
SEED = 781

INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
WORK_ROOT = Path('/kaggle/working/CS781_Capstone/phase2_improved') if Path('/kaggle/working').exists() else Path.cwd() / 'CS781_Capstone/phase2/run'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MODE = 'auto'  # auto resumes verified work, or starts fresh; inference_only is optional

OFFICIAL_LABELS = frozenset({
    'APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT',
    'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO',
})
DATA_NAMES = ('fold1-text.json', 'fold1-gold.json', 'fold2-text.json',
              'fold2-gold.json', 'labels.txt', 'fold3-text.json')
BASE_ARCHIVES = {f'{i:02d}': f'phase1_submission{i:02d}_recovery.zip' for i in range(1, 5)}
BASE_SOURCES = {
    '01': 'decoder_lm.py', '02': 'encoder_decoder_lm.py',
    '03': 'prefix_hybrid_lm.py', '04': 'efficient_causal_lm.py',
}
STAGED_BASES: dict[str, Path] = {}
RESTORED_RECOVERY_SLOTS: dict[int, Path] = {}

# Five distinct non-distillation methods and one trained-teacher distillation.
EXPERIMENTS = [
    {'slot': 1, 'base_id': '01', 'method': 'full_finetune', 'teacher_id': None,
     'module_filename': 'phase2_method_01.py'},
    {'slot': 2, 'base_id': '02', 'method': 'partial_finetune', 'teacher_id': None,
     'module_filename': 'phase2_method_02.py'},
    {'slot': 3, 'base_id': '03', 'method': 'lora', 'teacher_id': None,
     'module_filename': 'phase2_method_03.py'},
    {'slot': 4, 'base_id': '04', 'method': 'lorma', 'teacher_id': None,
     'module_filename': 'phase2_method_04.py'},
    {'slot': 5, 'base_id': '01', 'method': 'adapter', 'teacher_id': None,
     'module_filename': 'phase2_method_05.py'},
    {'slot': 6, 'base_id': '03', 'method': 'knowledge_distillation', 'teacher_id': '01',
     'module_filename': 'phase2_method_06.py'},
]

TRAINING_CONFIGS = {
    i: {
        'max_steps': (1000 if i in (3, 4, 6) else 800),
        'batch_size': (4 if i == 6 else 8), 'gradient_accumulation_steps': 1,
        'learning_rate': (2e-5 if i == 1 else 5e-5 if i == 2 else 2e-4),
        'weight_decay': 0.01, 'validation_interval': 400,
        'validation_document_cap': 64, 'max_tokens': 256, 'window_stride': 192,
        'gradient_clip_norm': 1.0, 'checkpoint_interval': 100,
        'recovery_interval': 400, 'head_dropout': .1,
        'outside_class_weight': .3, 'confidence_threshold': .45,
        'max_char_shift': 16, 'boundary_loss_weight': .2,
        'inference_window_batch_size': 4, 'unfrozen_layers': 2,
        'rank': 8, 'adapter_alpha': 16.0, 'adapter_bottleneck': 48,
        'teacher_max_tokens': 384, 'distill_temperature': 2.0,
        'distill_weight': .35,
        'optimizer': 'AdamW', 'seed': SEED + i,
    }
    for i in range(1, 7)
}
VALIDATION_FRACTION = 0.10
METHOD_SOURCE_OVERRIDES: dict[int, str] = {}
FILE_OVERRIDES: dict[str, str] = {}
RESUME_EXPECTED_SLOTS: set[int] | None = None  # optional stricter resume audit
AUTO_DOWNLOAD_PHASE2 = True
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
HF_DATASET_FILES = {
    'fold1-text.json': 'Phase_2_LNER/Train/fold1-text.json',
    'fold1-gold.json': 'Phase_2_LNER/Train/fold1-gold.json',
    'fold2-text.json': 'Phase_2_LNER/Train/fold2-text.json',
    'fold2-gold.json': 'Phase_2_LNER/Train/fold2-gold.json',
    'labels.txt': 'Phase_2_LNER/Train/labels.txt',
    'fold3-text.json': 'Phase_2_LNER/Test/fold3-text.json',
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def stable_json(value: Any) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))


def atomic_json(path: Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent,
                                     suffix='.json', delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
        handle.flush()
        os.fsync(handle.fileno())
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def append_jsonl(path: Path, value: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(stable_json(dict(value)) + '\n')
        handle.flush()
        os.fsync(handle.fileno())


def discover_exact(filename: str, *, required: bool = True) -> Path | None:
    """
    Find an exact input file.

    Special handling is included for Phase-1 recovery archives.

    Kaggle may expose:
        phase1_submission01_recovery.zip

    as an extracted dataset directory instead of an actual ZIP file.

    In that case, this function automatically creates a compatibility ZIP
    in /kaggle/working from the extracted dataset so that the rest of the
    Phase-2 notebook can continue unchanged.
    """

    # ============================================================
    # 1. HANDLE EXPLICIT FILE OVERRIDES
    # ============================================================

    if filename in FILE_OVERRIDES:
        selected = Path(
            FILE_OVERRIDES[filename]
        ).expanduser().resolve()

        if not selected.is_file():
            raise FileNotFoundError(
                f'Override file does not exist for {filename}: {selected}'
            )

        if selected.name != filename:
            raise FileNotFoundError(
                f'Override filename mismatch.\n'
                f'Expected: {filename}\n'
                f'Found:    {selected.name}\n'
                f'Path:     {selected}'
            )

        return selected


    # ============================================================
    # 2. NORMAL EXACT FILE SEARCH
    # ============================================================

    matches = sorted(
        set(
            path.resolve()
            for path in INPUT_ROOT.rglob(filename)
            if path.is_file()
        )
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise RuntimeError(
            f'Expected one {filename}; '
            f'found {len(matches)}: {matches}'
        )


    # ============================================================
    # 3. SPECIAL FALLBACK FOR PHASE-1 RECOVERY ARCHIVES
    # ============================================================

    import re

    archive_match = re.fullmatch(
        r'phase1_submission0?([1-4])_recovery\.zip',
        filename
    )

    if archive_match is not None:

        phase1_number = int(
            archive_match.group(1)
        )

        base_id = f'{phase1_number:02d}'

        source_filename_map = {
            '01': 'decoder_lm.py',
            '02': 'encoder_decoder_lm.py',
            '03': 'prefix_hybrid_lm.py',
            '04': 'efficient_causal_lm.py',
        }

        source_filename = source_filename_map[
            base_id
        ]


        # ========================================================
        # 4. FIND EXTRACTED KAGGLE PHASE-1 DATASET
        # ========================================================

        candidate_roots = []

        for source_path in INPUT_ROOT.rglob(
            source_filename
        ):

            if not source_path.is_file():
                continue

            # Expected structure:
            #
            # recovery_root/
            #     model/
            #         decoder_lm.py
            #
            if source_path.parent.name != 'model':
                continue

            recovery_root = (
                source_path.parent.parent.resolve()
            )

            required_files = [
                recovery_root
                / 'model'
                / source_filename,

                recovery_root
                / 'model'
                / 'course_bpe.py',

                recovery_root
                / 'checkpoints'
                / 'selected_model.pt',

                recovery_root
                / 'tokenizer'
                / 'tokenizer.json',

                recovery_root
                / 'tokenizer'
                / 'tokenizer_metadata.json',
            ]

            if all(
                path.is_file()
                for path in required_files
            ):
                candidate_roots.append(
                    recovery_root
                )


        candidate_roots = sorted(
            set(candidate_roots)
        )


        # ========================================================
        # 5. IF MULTIPLE DATASETS MATCH, PREFER CORRECT NAME
        # ========================================================

        if len(candidate_roots) > 1:

            def normalized_path_name(path: Path) -> str:
                return re.sub(
                    r'[^a-z0-9]',
                    '',
                    str(path).lower()
                )

            accepted_tokens = {
                f'phase1submission{base_id}recovery',
                f'phase1submission{phase1_number}recovery',
            }

            preferred_candidates = []

            for candidate in candidate_roots:

                normalized = normalized_path_name(
                    candidate
                )

                if any(
                    token in normalized
                    for token in accepted_tokens
                ):
                    preferred_candidates.append(
                        candidate
                    )

            if len(preferred_candidates) == 1:
                candidate_roots = (
                    preferred_candidates
                )


        # ========================================================
        # 6. EXACTLY ONE EXTRACTED PHASE-1 DATASET FOUND
        # ========================================================

        if len(candidate_roots) == 1:

            recovery_root = (
                candidate_roots[0]
            )

            compatibility_directory = (
                WORK_ROOT
                / '_phase1_archive_compat'
            )

            compatibility_directory.mkdir(
                parents=True,
                exist_ok=True
            )

            compatibility_zip = (
                compatibility_directory
                / filename
            )


            # ====================================================
            # 7. CREATE COMPATIBILITY ZIP IF NEEDED
            # ====================================================

            if not compatibility_zip.exists():

                print()
                print(
                    f'[Phase-1 {base_id}] '
                    f'Original recovery ZIP not found.'
                )

                print(
                    f'[Phase-1 {base_id}] '
                    f'Using extracted Kaggle dataset:'
                )

                print(
                    f'    {recovery_root}'
                )

                print(
                    f'[Phase-1 {base_id}] '
                    f'Creating compatibility archive:'
                )

                print(
                    f'    {compatibility_zip}'
                )


                # ================================================
                # FILES REQUIRED BY PHASE-2
                # ================================================

                files_to_add = [
                    (
                        recovery_root
                        / 'model'
                        / source_filename,

                        f'model/{source_filename}'
                    ),

                    (
                        recovery_root
                        / 'model'
                        / 'course_bpe.py',

                        'model/course_bpe.py'
                    ),

                    (
                        recovery_root
                        / 'checkpoints'
                        / 'selected_model.pt',

                        'checkpoints/selected_model.pt'
                    ),
                ]


                # ================================================
                # INCLUDE ENTIRE TOKENIZER DIRECTORY
                # ================================================

                tokenizer_root = (
                    recovery_root
                    / 'tokenizer'
                )

                if not tokenizer_root.is_dir():
                    raise RuntimeError(
                        f'Phase-1 {base_id}: '
                        f'tokenizer directory missing:\n'
                        f'{tokenizer_root}'
                    )

                for tokenizer_file in sorted(
                    tokenizer_root.rglob('*')
                ):

                    if not tokenizer_file.is_file():
                        continue

                    archive_name = (
                        tokenizer_file
                        .relative_to(recovery_root)
                        .as_posix()
                    )

                    files_to_add.append(
                        (
                            tokenizer_file,
                            archive_name
                        )
                    )


                # ================================================
                # REMOVE POSSIBLE DUPLICATES
                # ================================================

                unique_files = {}

                for source_file, archive_name in files_to_add:
                    unique_files[
                        archive_name
                    ] = source_file


                # ================================================
                # TEMPORARY ZIP FOR SAFE ATOMIC CREATION
                # ================================================

                temporary_zip = (
                    compatibility_directory
                    / f'{filename}.tmp'
                )

                temporary_zip.unlink(
                    missing_ok=True
                )


                try:

                    # ============================================
                    # CREATE ZIP
                    # ============================================

                    with zipfile.ZipFile(
                        temporary_zip,
                        mode='w',
                        compression=zipfile.ZIP_DEFLATED,
                        allowZip64=True,
                    ) as archive:

                        for (
                            archive_name,
                            source_file
                        ) in sorted(
                            unique_files.items()
                        ):

                            if not source_file.is_file():
                                raise FileNotFoundError(
                                    f'Missing required Phase-1 file:\n'
                                    f'{source_file}'
                                )

                            archive.write(
                                source_file,
                                arcname=archive_name
                            )


                    # ============================================
                    # VERIFY CREATED ZIP
                    # ============================================

                    with zipfile.ZipFile(
                        temporary_zip,
                        mode='r'
                    ) as archive:

                        bad_member = (
                            archive.testzip()
                        )

                        if bad_member is not None:
                            raise RuntimeError(
                                f'Phase-1 {base_id}: '
                                f'ZIP CRC check failed for '
                                f'{bad_member}'
                            )

                        archive_names = set(
                            archive.namelist()
                        )

                        required_archive_files = {
                            f'model/{source_filename}',
                            'model/course_bpe.py',
                            'checkpoints/selected_model.pt',
                            'tokenizer/tokenizer.json',
                            'tokenizer/tokenizer_metadata.json',
                        }

                        missing_archive_files = (
                            required_archive_files
                            - archive_names
                        )

                        if missing_archive_files:
                            raise RuntimeError(
                                f'Phase-1 {base_id}: '
                                f'compatibility ZIP is missing:\n'
                                f'{sorted(missing_archive_files)}'
                            )


                    # ============================================
                    # ATOMICALLY MOVE INTO FINAL LOCATION
                    # ============================================

                    os.replace(
                        temporary_zip,
                        compatibility_zip
                    )


                finally:

                    temporary_zip.unlink(
                        missing_ok=True
                    )


            # ====================================================
            # 8. VERIFY EXISTING/CREATED COMPATIBILITY ZIP
            # ====================================================

            try:

                with zipfile.ZipFile(
                    compatibility_zip,
                    mode='r'
                ) as archive:

                    bad_member = (
                        archive.testzip()
                    )

                    if bad_member is not None:
                        raise RuntimeError(
                            f'Phase-1 {base_id}: '
                            f'cached compatibility ZIP '
                            f'is corrupt at {bad_member}'
                        )

                    archive_names = set(
                        archive.namelist()
                    )

                    required_archive_files = {
                        f'model/{source_filename}',
                        'model/course_bpe.py',
                        'checkpoints/selected_model.pt',
                        'tokenizer/tokenizer.json',
                        'tokenizer/tokenizer_metadata.json',
                    }

                    missing_archive_files = (
                        required_archive_files
                        - archive_names
                    )

                    if missing_archive_files:
                        raise RuntimeError(
                            f'Phase-1 {base_id}: '
                            f'cached compatibility ZIP '
                            f'is missing:\n'
                            f'{sorted(missing_archive_files)}'
                        )


            except Exception:

                compatibility_zip.unlink(
                    missing_ok=True
                )

                raise


            print(
                f'[Phase-1 {base_id}] '
                f'Compatibility archive ready:'
            )

            print(
                f'    {compatibility_zip}'
            )

            print()

            return compatibility_zip.resolve()


        # ========================================================
        # 9. MULTIPLE POSSIBLE RECOVERY DIRECTORIES FOUND
        # ========================================================

        if len(candidate_roots) > 1:

            candidates_text = '\n'.join(
                f'    {path}'
                for path in candidate_roots
            )

            raise RuntimeError(
                f'Could not uniquely identify '
                f'Phase-1 submission {base_id}.\n'
                f'Multiple valid extracted recovery '
                f'directories were found:\n\n'
                f'{candidates_text}'
            )


        # ========================================================
        # 10. NO VALID EXTRACTED RECOVERY FOUND
        # ========================================================

        if required:

            raise RuntimeError(
                f'Could not find Phase-1 submission '
                f'{base_id}.\n\n'
                f'Expected either:\n\n'
                f'1. {filename}\n\n'
                f'OR an extracted Kaggle dataset containing:\n\n'
                f'    model/{source_filename}\n'
                f'    model/course_bpe.py\n'
                f'    checkpoints/selected_model.pt\n'
                f'    tokenizer/tokenizer.json\n'
                f'    tokenizer/tokenizer_metadata.json\n\n'
                f'Searched inside:\n'
                f'    {INPUT_ROOT}'
            )

        return None


    # ============================================================
    # 11. NORMAL NON-PHASE-1 FILE NOT FOUND
    # ============================================================

    if required:
        raise RuntimeError(
            f'Expected one {filename}; '
            f'found 0: []'
        )

    return None


def stage_official_phase2_files() -> None:
    """Fill missing files and verify attached ones against one HF revision."""
    needed = ('labels.txt', 'fold3-text.json') if RUN_MODE == 'inference_only' else DATA_NAMES
    missing = [name for name in needed if discover_exact(name, required=False) is None]
    if not missing or not AUTO_DOWNLOAD_PHASE2:
        return
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError(
            'Missing Phase-2 files. Add an enabled Kaggle secret named HF_TOKEN, '
            'accept access to the gated Hugging Face course dataset, and enable Internet; '
            'or attach the six official files as Kaggle Inputs.'
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError('Kaggle secret HF_TOKEN is empty; add a Hugging Face read token')
    token = token.strip()
    try:
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                   'huggingface_hub>=0.24,<2'])
            from huggingface_hub import HfApi, hf_hub_download
        except Exception:
            raise RuntimeError(
                'huggingface_hub is unavailable. Enable Kaggle Internet or attach the official files.'
            ) from None
    try:
        revision = HfApi().repo_info(HF_DATASET_REPO, repo_type='dataset', token=token).sha
        if not isinstance(revision, str) or len(revision) != 40:
            raise ValueError('Hugging Face did not return a full dataset commit')
        manifest_path = WORK_ROOT / 'official_phase2_source.json'
        if manifest_path.exists():
            existing = json.loads(manifest_path.read_text(encoding='utf-8'))
            if existing.get('repo_id') != HF_DATASET_REPO or existing.get('revision') != revision:
                raise RuntimeError('Official dataset revision changed in this workspace; refusing mixed folds')
        destination = WORK_ROOT / 'official_phase2'
        destination.mkdir(parents=True, exist_ok=True)
        records = {}
        for name in needed:
            cached = Path(hf_hub_download(
                repo_id=HF_DATASET_REPO, repo_type='dataset',
                filename=HF_DATASET_FILES[name], revision=revision,
                token=token, cache_dir=str(WORK_ROOT / 'huggingface_cache')))
            if not cached.is_file() or cached.stat().st_size == 0:
                raise RuntimeError(f'Empty or missing Hugging Face file: {name}')
            digest = sha256_file(cached)
            if name not in missing:
                target = discover_exact(name)
                if sha256_file(target) != digest:
                    raise RuntimeError(f'Attached {name} differs from the pinned official dataset revision')
            else:
                target = destination / name
                if target.exists():
                    if sha256_file(target) != digest:
                        raise RuntimeError(f'Conflicting Phase-2 file in workspace: {name}')
                else:
                    with tempfile.NamedTemporaryFile(dir=destination, delete=False) as handle:
                        temporary = Path(handle.name)
                    try:
                        shutil.copyfile(cached, temporary)
                        if sha256_file(temporary) != digest:
                            raise RuntimeError(f'Phase-2 download copy failed integrity check: {name}')
                        os.replace(temporary, target)
                    finally:
                        temporary.unlink(missing_ok=True)
                FILE_OVERRIDES[name] = str(target)
            records[name] = {'source_path': HF_DATASET_FILES[name],
                             'bytes': target.stat().st_size, 'sha256': digest}
        prior_files = existing.get('files', {}) if manifest_path.exists() else {}
        atomic_json(manifest_path, {'repo_id': HF_DATASET_REPO, 'revision': revision,
                                    'files': {**prior_files, **records}})
        print(f'Fetched {len(missing)} official Phase-2 file(s) from Hugging Face revision {revision[:12]}.')
    except RuntimeError:
        raise
    except Exception:
        raise RuntimeError(
            'Phase-2 Hugging Face download failed. Check HF_TOKEN access to the gated dataset '
            'and Kaggle Internet, or attach the six official files as Kaggle Inputs.'
        ) from None
    finally:
        del token


# Prefer the verified files already present in the previous Kaggle working tree.
OLD_PHASE2_ROOT = Path('/kaggle/working/CS781_Capstone/phase2')
for _name in DATA_NAMES:
    _existing = OLD_PHASE2_ROOT / 'official_phase2' / _name
    if _existing.is_file():
        FILE_OVERRIDES[_name] = str(_existing)

if RUN_PHASE2:
    stage_official_phase2_files()


FOUND = {name: discover_exact(name, required=False) for name in (*DATA_NAMES, *BASE_ARCHIVES.values())}
print('Detected official files and Phase-1 archives:')
for name, path in FOUND.items():
    print(f'  {name}: {path if path else "not attached"}')
print('CUDA devices:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])


In [ ]:
# ================================================================
# PHASE-1 ARCHIVE VERIFICATION + SAFE SELECTIVE EXTRACTION
# Supports original recovery ZIPs AND compatibility ZIPs created
# from Kaggle-extracted Phase-1 recovery datasets.
# ================================================================


def _safe_archive_member(name: str) -> bool:
    return (
        bool(name)
        and not name.startswith('/')
        and '\\' not in name
        and '..' not in Path(name).parts
    )


def _integrity_rows(
    archive: zipfile.ZipFile
) -> dict[str, dict[str, Any]] | None:

    name = 'manifests/recovery_integrity.json'

    if name not in archive.namelist():
        return None

    manifest = json.loads(
        archive.read(name)
    )

    rows = {
        row['path']: row
        for row in manifest['files']
    }

    if (
        len(rows) != len(manifest['files'])
        or set(rows)
        != set(archive.namelist()) - {name}
    ):
        raise ValueError(
            'Archive integrity manifest does not match ZIP members'
        )

    return rows


def extract_member_verified(
    archive: zipfile.ZipFile,
    member: str,
    destination: Path,
    integrity: Mapping[str, Any] | None,
) -> dict[str, Any]:

    if (
        not _safe_archive_member(member)
        or member not in archive.namelist()
    ):
        raise ValueError(
            f'Missing/unsafe required archive member: {member}'
        )

    info = archive.getinfo(member)

    if info.is_dir():
        raise ValueError(
            f'Unexpected directory member: {member}'
        )

    if info.file_size > 1_500_000_000:
        raise ValueError(
            f'Archive member is unexpectedly large: {member}'
        )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    digest = hashlib.sha256()

    with tempfile.NamedTemporaryFile(
        dir=destination.parent,
        delete=False,
    ) as handle:

        temporary = Path(handle.name)

        try:

            with archive.open(info) as source:

                while True:

                    chunk = source.read(1 << 20)

                    if not chunk:
                        break

                    digest.update(chunk)
                    handle.write(chunk)

            handle.flush()
            os.fsync(handle.fileno())

        except Exception:

            temporary.unlink(
                missing_ok=True
            )

            raise

    actual = digest.hexdigest()

    try:

        # --------------------------------------------------------
        # Verify recovery-integrity manifest when original archive
        # contains one.
        # --------------------------------------------------------

        if integrity is not None:

            row = integrity.get(member)

            if row is None:
                raise ValueError(
                    f'Integrity manifest has no entry for {member}'
                )

            if row.get('sha256') != actual:
                raise ValueError(
                    f'Integrity SHA256 mismatch: {member}'
                )

            if int(
                row.get('bytes', -1)
            ) != info.file_size:
                raise ValueError(
                    f'Integrity size mismatch: {member}'
                )

        # --------------------------------------------------------
        # IMPORTANT:
        # If an old staged file exists and is identical, keep it.
        #
        # If it differs, replace the staged copy with the artifact
        # from the currently attached recovery dataset.
        #
        # This prevents stale /kaggle/working files from breaking
        # a new run.
        # --------------------------------------------------------

        if destination.exists():

            existing_hash = sha256_file(
                destination
            )

            if existing_hash != actual:

                print(
                    f'Replacing stale staged artifact: '
                    f'{destination}'
                )

                os.replace(
                    temporary,
                    destination
                )

            else:

                temporary.unlink(
                    missing_ok=True
                )

        else:

            os.replace(
                temporary,
                destination
            )

    finally:

        temporary.unlink(
            missing_ok=True
        )

    return {
        'member': member,
        'sha256': actual,
        'bytes': info.file_size,
        'manifest_verified': (
            integrity is not None
        ),
    }


def stage_phase1_base(
    base_id: str
) -> Path:

    # ------------------------------------------------------------
    # If already staged successfully THIS Python session,
    # simply reuse it.
    # ------------------------------------------------------------

    if base_id in STAGED_BASES:

        staged = STAGED_BASES[
            base_id
        ]

        required_existing = [
            staged
            / 'model'
            / BASE_SOURCES[base_id],

            staged
            / 'model'
            / 'course_bpe.py',

            staged
            / 'checkpoints'
            / 'selected_model.pt',

            staged
            / 'tokenizer'
            / 'tokenizer.json',

            staged
            / 'tokenizer'
            / 'tokenizer_metadata.json',
        ]

        if all(
            path.is_file()
            for path in required_existing
        ):
            return staged

        # Cached entry is incomplete.
        STAGED_BASES.pop(
            base_id,
            None
        )


    archive_name = BASE_ARCHIVES[
        base_id
    ]

    archive_path = discover_exact(
        archive_name
    )

    destination = (
        WORK_ROOT
        / 'phase1_bases'
        / base_id
    )

    source = BASE_SOURCES[
        base_id
    ]


    print()
    print(
        f'[Phase-1 {base_id}] '
        f'Staging base model from:'
    )
    print(
        f'    {archive_path}'
    )


    # ------------------------------------------------------------
    # Required Phase-1 members
    # ------------------------------------------------------------

    required_members = [
        f'model/{source}',
        'model/course_bpe.py',
        'checkpoints/selected_model.pt',
        'tokenizer/tokenizer.json',
        'tokenizer/tokenizer_metadata.json',
    ]


    with zipfile.ZipFile(
        archive_path,
        mode='r'
    ) as archive:

        # --------------------------------------------------------
        # CRC TEST FIRST
        # --------------------------------------------------------

        bad_member = archive.testzip()

        if bad_member is not None:
            raise RuntimeError(
                f'Phase-1 {base_id}: '
                f'ZIP CRC failure at {bad_member}'
            )


        names = archive.namelist()


        # --------------------------------------------------------
        # Safety checks
        # --------------------------------------------------------

        if len(names) != len(set(names)):
            raise ValueError(
                f'Phase-1 {base_id}: '
                f'duplicate ZIP member names'
            )

        unsafe = [
            name
            for name in names
            if not _safe_archive_member(name)
        ]

        if unsafe:
            raise ValueError(
                f'Phase-1 {base_id}: '
                f'unsafe ZIP members: {unsafe}'
            )


        # --------------------------------------------------------
        # Make sure required artifacts exist
        # --------------------------------------------------------

        missing = [
            member
            for member in required_members
            if member not in names
        ]

        if missing:

            raise RuntimeError(
                f'Phase-1 {base_id}: '
                f'recovery archive is missing required files:\n'
                + '\n'.join(
                    f'    {item}'
                    for item in missing
                )
            )


        # --------------------------------------------------------
        # Include all tokenizer files.
        # Some tokenizer implementations require additional files.
        # --------------------------------------------------------

        tokenizer_members = [
            name
            for name in names
            if (
                name.startswith(
                    'tokenizer/'
                )
                and not name.endswith('/')
            )
        ]


        members = [
            f'model/{source}',
            'model/course_bpe.py',
            'checkpoints/selected_model.pt',
        ]

        members.extend(
            tokenizer_members
        )

        # Remove duplicates while preserving order.
        members = list(
            dict.fromkeys(members)
        )


        integrity = _integrity_rows(
            archive
        )


        records = []

        for member in members:

            record = extract_member_verified(
                archive,
                member,
                destination / member,
                integrity,
            )

            records.append(
                record
            )


    # ------------------------------------------------------------
    # IMPORTANT FIX
    #
    # Do NOT bind the staged Phase-1 model to the physical ZIP hash.
    #
    # Original ZIP and compatibility ZIP may have different ZIP
    # hashes despite containing byte-identical model artifacts.
    #
    # Instead bind provenance to hashes of the ACTUAL artifacts.
    # ------------------------------------------------------------

    selected_hashes = {
        record['member']:
        record['sha256']
        for record in records
    }


    evidence = {
        'base_id': base_id,
        'archive_name': archive_name,
        'selected_members': records,
        'selected_member_hashes': selected_hashes,
        'assurance': (
            'manifest+SHA256+ZIP-CRC'
            if integrity is not None
            else
            'local-SHA256+ZIP-CRC'
        ),
    }


    manifest_path = (
        destination
        / 'base_evidence.json'
    )


    # ------------------------------------------------------------
    # Old notebook versions may have written evidence containing
    # archive_sha256. That should NOT cause a failure.
    #
    # We safely overwrite evidence after all member hashes have
    # been independently verified above.
    # ------------------------------------------------------------

    atomic_json(
        manifest_path,
        evidence
    )


    STAGED_BASES[
        base_id
    ] = destination


    print(
        f'[Phase-1 {base_id}] '
        f'Base staged successfully.'
    )

    print(
        f'    model: '
        f'{destination / "model" / source}'
    )

    print(
        f'    checkpoint: '
        f'{destination / "checkpoints" / "selected_model.pt"}'
    )

    print(
        f'    tokenizer: '
        f'{destination / "tokenizer"}'
    )

    print()

    return destination


def import_local_source(
    path: Path,
    name: str
):

    spec = (
        importlib.util
        .spec_from_file_location(
            name,
            path
        )
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise ImportError(
            path
        )

    module = (
        importlib.util
        .module_from_spec(
            spec
        )
    )

    sys.modules[
        name
    ] = module

    spec.loader.exec_module(
        module
    )

    return module


def load_phase1_base(
    base_id: str,
    device: str = 'cpu'
):

    # ============================================================
    # STAGE PHASE-1 ARTIFACTS
    # ============================================================

    directory = stage_phase1_base(
        base_id
    )


    source_path = (
        directory
        / 'model'
        / BASE_SOURCES[base_id]
    )


    bpe_source_path = (
        directory
        / 'model'
        / 'course_bpe.py'
    )


    checkpoint_path = (
        directory
        / 'checkpoints'
        / 'selected_model.pt'
    )


    tokenizer_directory = (
        directory
        / 'tokenizer'
    )


    tokenizer_metadata_path = (
        tokenizer_directory
        / 'tokenizer_metadata.json'
    )


    # ============================================================
    # VERIFY REQUIRED FILES
    # ============================================================

    required_paths = [
        source_path,
        bpe_source_path,
        checkpoint_path,
        tokenizer_directory
        / 'tokenizer.json',
        tokenizer_metadata_path,
    ]


    missing_paths = [
        path
        for path in required_paths
        if not path.is_file()
    ]


    if missing_paths:

        raise FileNotFoundError(
            f'Phase-1 {base_id}: '
            f'staged base is incomplete:\n'
            + '\n'.join(
                f'    {path}'
                for path in missing_paths
            )
        )


    # ============================================================
    # IMPORT PHASE-1 MODEL SOURCE
    # ============================================================

    model_module_name = (
        f'cs781_phase1_'
        f'{base_id}_'
        f'{sha256_file(source_path)[:12]}'
    )


    module = import_local_source(
        source_path,
        model_module_name,
    )


    # ============================================================
    # IMPORT TOKENIZER SOURCE
    # ============================================================

    tokenizer_module_name = (
        f'cs781_bpe_'
        f'{base_id}_'
        f'{sha256_file(bpe_source_path)[:12]}'
    )


    tokenizer_module = import_local_source(
        bpe_source_path,
        tokenizer_module_name,
    )


    if not hasattr(
        tokenizer_module,
        'CourseBPETokenizer'
    ):
        raise AttributeError(
            f'Phase-1 {base_id}: '
            f'course_bpe.py does not define '
            f'CourseBPETokenizer'
        )


    # ============================================================
    # RESTORE TOKENIZER
    # ============================================================

    tokenizer = (
        tokenizer_module
        .CourseBPETokenizer
        .from_pretrained(
            tokenizer_directory
        )
    )


    # ============================================================
    # LOAD CHECKPOINT
    # ============================================================

    checkpoint = torch.load(
        checkpoint_path,
        map_location='cpu',
        weights_only=True,
    )


    if not isinstance(
        checkpoint,
        Mapping
    ):
        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'invalid selected_model.pt'
        )


    if 'model_config' not in checkpoint:
        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'checkpoint has no model_config'
        )


    if 'model_state' not in checkpoint:
        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'checkpoint has no model_state'
        )


    # ============================================================
    # VERIFY MODEL SOURCE BINDING
    # ============================================================

    recorded_source_sha = (
        checkpoint.get(
            'model_source_sha256'
        )
    )


    actual_source_sha = (
        sha256_file(
            source_path
        )
    )


    if (
        recorded_source_sha is not None
        and recorded_source_sha
        != actual_source_sha
    ):

        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'checkpoint/model source SHA mismatch.\n'
            f'Checkpoint expects:\n'
            f'    {recorded_source_sha}\n'
            f'Actual source:\n'
            f'    {actual_source_sha}'
        )


    # Preserve the notebook's original special handling for
    # Phase-1 submission 01.
    if (
        recorded_source_sha is None
        and base_id != '01'
    ):

        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'checkpoint has no model-source binding'
        )


    # ============================================================
    # VERIFY TOKENIZER FINGERPRINT
    # ============================================================

    tokenizer_meta = json.loads(
        tokenizer_metadata_path.read_text(
            encoding='utf-8'
        )
    )


    checkpoint_tokenizer_fingerprint = (
        checkpoint.get(
            'tokenizer_fingerprint'
        )
    )


    metadata_tokenizer_fingerprint = (
        tokenizer_meta.get(
            'fingerprint'
        )
    )


    if (
        checkpoint_tokenizer_fingerprint
        != metadata_tokenizer_fingerprint
    ):

        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'tokenizer fingerprint mismatch.\n'
            f'Checkpoint:\n'
            f'    {checkpoint_tokenizer_fingerprint}\n'
            f'Tokenizer metadata:\n'
            f'    {metadata_tokenizer_fingerprint}'
        )


    # ============================================================
    # VERIFY VOCABULARY SIZE
    # ============================================================

    checkpoint_vocab_size = int(
        checkpoint[
            'model_config'
        ][
            'vocab_size'
        ]
    )


    tokenizer_vocab_size = len(
        tokenizer
    )


    if (
        checkpoint_vocab_size
        != tokenizer_vocab_size
    ):

        raise RuntimeError(
            f'Phase-1 {base_id}: '
            f'vocabulary mismatch.\n'
            f'Checkpoint vocab size: '
            f'{checkpoint_vocab_size}\n'
            f'Tokenizer vocab size: '
            f'{tokenizer_vocab_size}'
        )


    # ============================================================
    # BUILD MODEL
    # ============================================================

    if not callable(
        getattr(
            module,
            'build_model',
            None
        )
    ):

        raise AttributeError(
            f'Phase-1 {base_id}: '
            f'model source does not define build_model()'
        )


    model = module.build_model(
        checkpoint[
            'model_config'
        ]
    )


    # ============================================================
    # LOAD TRAINED WEIGHTS
    # ============================================================

    model.load_state_dict(
        checkpoint[
            'model_state'
        ],
        strict=True,
    )


    parameter_count = sum(
        parameter.numel()
        for parameter
        in model.parameters()
    )


    model = model.to(
        device
    )


    # ============================================================
    # RETURN MODEL + TOKENIZER + VERIFIED METADATA
    # ============================================================

    metadata = {
        'base_id': base_id,

        'checkpoint_sha256':
            sha256_file(
                checkpoint_path
            ),

        'source_sha256':
            actual_source_sha,

        'source_binding':
            (
                'checkpoint'
                if recorded_source_sha
                is not None
                else
                'same_archive_only'
            ),

        'tokenizer_fingerprint':
            metadata_tokenizer_fingerprint,

        'parameter_count':
            parameter_count,

        'checkpoint_step':
            checkpoint.get(
                'step'
            ),

        'model_config':
            checkpoint[
                'model_config'
            ],
    }


    print(
        f'[Phase-1 {base_id}] '
        f'Loaded successfully '
        f'({parameter_count:,} parameters)'
    )


    return (
        model,
        tokenizer,
        metadata,
    )

In [ ]:
# ================================================================
# OFFICIAL PHASE-2 DATASET LOADING + VALIDATION
#
# Robust to:
#   1. {"doc_id": "text"}
#   2. {"doc_id": {"text": "text"}}
#   3. harmless top-level metadata entries
#
# IMPORTANT:
# We never concatenate arbitrary pieces of text because doing so
# could change official character offsets.
# ================================================================


def load_json(path: Path) -> Any:
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f'JSON file does not exist: {path}'
        )

    with path.open(
        'r',
        encoding='utf-8'
    ) as handle:
        return json.load(handle)


# ================================================================
# TEXT VALUE NORMALIZATION
# ================================================================

def _extract_document_text(
    identifier: str,
    value: Any,
    *,
    path: Path,
) -> str | None:
    """
    Convert one official text-file value to a plain document string.

    Supported forms:

        "123": "document text"

    and

        "123": {
            "text": "document text"
        }

    A few equivalent wrapper keys are accepted defensively.

    Returns None only for recognizable metadata entries.
    """

    # ------------------------------------------------------------
    # NORMAL CASE:
    # ID -> STRING
    # ------------------------------------------------------------

    if isinstance(value, str):

        if not value:
            raise ValueError(
                f'{path}: document {identifier!r} '
                f'contains an empty text string'
            )

        return value


    # ------------------------------------------------------------
    # TEST FILE MAY WRAP TEXT IN AN OBJECT
    # ------------------------------------------------------------

    if isinstance(value, dict):

        text_keys = (
            'text',
            'document',
            'document_text',
            'content',
            'body',
            'input_text',
        )

        for key in text_keys:

            candidate = value.get(key)

            if (
                isinstance(candidate, str)
                and candidate
            ):
                return candidate


        # --------------------------------------------------------
        # If the object contains EXACTLY ONE nonempty string field,
        # use it. This safely handles wrappers whose field has an
        # unexpected name without concatenating anything.
        # --------------------------------------------------------

        string_values = [
            candidate
            for candidate in value.values()
            if (
                isinstance(candidate, str)
                and candidate
            )
        ]

        if len(string_values) == 1:
            return string_values[0]


    # ------------------------------------------------------------
    # KNOWN/SANE METADATA KEYS CAN BE IGNORED
    # ------------------------------------------------------------

    normalized_id = (
        str(identifier)
        .strip()
        .lower()
    )

    metadata_names = {
        'metadata',
        '_metadata',
        '__metadata__',
        'meta',
        '_meta',
        '__meta__',
        'info',
        '_info',
        '__info__',
    }

    if (
        normalized_id in metadata_names
        or normalized_id.startswith('__')
    ):
        print(
            f'Ignoring metadata entry '
            f'{identifier!r} in {path.name}'
        )

        return None


    # ------------------------------------------------------------
    # DO NOT GUESS BY CONCATENATING ARRAYS/OBJECTS.
    #
    # Character offsets depend on the exact source string.
    # ------------------------------------------------------------

    preview = repr(value)

    if len(preview) > 500:
        preview = preview[:500] + ' ...'

    raise ValueError(
        f'Unsupported text representation in {path}\n'
        f'Document ID: {identifier!r}\n'
        f'Value type: {type(value).__name__}\n'
        f'Value preview: {preview}'
    )


# ================================================================
# LOAD ID -> TEXT MAP
# ================================================================

def load_text_map(
    path: Path
) -> dict[str, str]:

    path = Path(path)

    raw = load_json(
        path
    )


    if (
        not isinstance(raw, dict)
        or not raw
    ):
        raise ValueError(
            f'Expected nonempty ID-to-text object: {path}'
        )


    result: dict[str, str] = {}

    direct_string_count = 0
    wrapped_string_count = 0
    ignored_count = 0


    for raw_identifier, raw_value in raw.items():

        identifier = str(
            raw_identifier
        )


        # Remember input representation only for useful diagnostics.
        was_direct_string = isinstance(
            raw_value,
            str
        )


        text = _extract_document_text(
            identifier,
            raw_value,
            path=path,
        )


        if text is None:
            ignored_count += 1
            continue


        if identifier in result:
            raise ValueError(
                f'Duplicate document ID '
                f'{identifier!r} in {path}'
            )


        if not isinstance(
            text,
            str
        ):
            raise TypeError(
                f'{identifier}: internal text-normalization '
                f'failure in {path}'
            )


        if len(text) == 0:
            raise ValueError(
                f'{identifier}: empty document in {path}'
            )


        result[
            identifier
        ] = text


        if was_direct_string:
            direct_string_count += 1
        else:
            wrapped_string_count += 1


    if not result:
        raise ValueError(
            f'No documents were found in {path}'
        )


    print(
        f'Loaded {len(result)} documents from {path.name} '
        f'({direct_string_count} direct strings, '
        f'{wrapped_string_count} wrapped texts'
        + (
            f', {ignored_count} metadata entries ignored'
            if ignored_count
            else ''
        )
        + ')'
    )


    return result


# ================================================================
# OFFICIAL LABEL VOCABULARY
# ================================================================

def load_label_vocabulary(
    path: Path
) -> list[str]:

    labels = [
        line.strip()
        for line
        in Path(path)
        .read_text(
            encoding='utf-8-sig'
        )
        .splitlines()
        if line.strip()
    ]


    if not labels:
        raise ValueError(
            f'No labels found in {path}'
        )


    if len(labels) != len(
        set(labels)
    ):
        raise ValueError(
            f'Duplicate labels in {path}: {labels}'
        )


    if set(labels) != OFFICIAL_LABELS:
        raise ValueError(
            f'Official Phase-2 labels differ '
            f'from expected labels: {labels}'
        )


    return labels


# ================================================================
# NORMALIZE ONE NER SPAN
# ================================================================

def normalize_span(
    raw: Any,
    *,
    document_id: str,
    text: str,
    allowed: set[str],
) -> tuple[int, int, str]:

    if (
        isinstance(
            raw,
            (tuple, list)
        )
        and len(raw) in (
            3,
            4,
        )
    ):

        start, end, label = raw[:3]

        supplied_text = (
            raw[3]
            if len(raw) == 4
            else None
        )


    elif isinstance(
        raw,
        dict
    ):

        start = raw.get(
            'start',
            raw.get(
                'start_offset'
            )
        )

        end = raw.get(
            'end',
            raw.get(
                'end_offset'
            )
        )

        label = raw.get(
            'label',
            raw.get(
                'entity_type'
            )
        )

        supplied_text = raw.get(
            'entity_text',
            raw.get(
                'text'
            )
        )


    else:

        raise TypeError(
            f'{document_id}: unsupported gold span '
            f'{raw!r}'
        )


    if (
        isinstance(start, bool)
        or isinstance(end, bool)
        or not isinstance(start, int)
        or not isinstance(end, int)
    ):

        raise ValueError(
            f'{document_id}: character offsets '
            f'must be integers: {raw!r}'
        )


    if not (
        0 <= start < end <= len(text)
    ):

        raise ValueError(
            f'{document_id}: invalid character offsets '
            f'{start}:{end} for document length {len(text)}'
        )


    if label not in allowed:

        raise ValueError(
            f'{document_id}: invalid entity label '
            f'{label!r}: {raw!r}'
        )


    if supplied_text is not None:

        if not isinstance(
            supplied_text,
            str
        ):
            raise ValueError(
                f'{document_id}: supplied entity text '
                f'is not a string: {raw!r}'
            )

        actual_text = text[
            start:end
        ]

        if actual_text != supplied_text:

            raise ValueError(
                f'{document_id}: gold entity text differs '
                f'from text[start:end].\n'
                f'Offsets: {start}:{end}\n'
                f'Expected from source: {actual_text!r}\n'
                f'Gold supplied text: {supplied_text!r}'
            )


    return (
        start,
        end,
        str(label),
    )


# ================================================================
# LOAD GOLD ANNOTATIONS
# ================================================================

def load_gold_map(
    path: Path,
    texts: Mapping[str, str],
    labels: Sequence[str],
) -> dict[
    str,
    list[
        tuple[
            int,
            int,
            str,
        ]
    ]
]:

    raw = load_json(
        path
    )


    if not isinstance(
        raw,
        dict
    ):
        raise ValueError(
            f'Expected ID-to-gold object: {path}'
        )


    raw_ids = {
        str(identifier)
        for identifier
        in raw
    }


    if raw_ids != set(
        texts
    ):

        missing = sorted(
            set(texts)
            - raw_ids
        )

        extra = sorted(
            raw_ids
            - set(texts)
        )

        raise ValueError(
            f'Gold IDs must match text IDs exactly: {path}\n'
            f'Missing gold IDs: {missing[:10]}\n'
            f'Extra gold IDs: {extra[:10]}'
        )


    allowed = set(
        labels
    )


    result = {}


    for identifier, raw_spans in raw.items():

        identifier = str(
            identifier
        )


        if not isinstance(
            raw_spans,
            list
        ):
            raise TypeError(
                f'{identifier}: expected a list '
                f'of gold entities'
            )


        spans = [
            normalize_span(
                item,
                document_id=identifier,
                text=texts[
                    identifier
                ],
                allowed=allowed,
            )
            for item
            in raw_spans
        ]


        if len(spans) != len(
            set(spans)
        ):
            raise ValueError(
                f'{identifier}: duplicate gold spans'
            )


        result[
            identifier
        ] = sorted(
            spans
        )


    return result


# ================================================================
# LOAD COMPLETE OFFICIAL PHASE-2 DATASET
# ================================================================

def load_official_phase2() -> dict[
    str,
    Any
]:

    paths = {
        name:
        discover_exact(
            name
        )
        for name
        in DATA_NAMES
    }


    labels = load_label_vocabulary(
        paths[
            'labels.txt'
        ]
    )


    train_text = {}
    train_gold = {}


    # ------------------------------------------------------------
    # FOLD 1 + FOLD 2 = PUBLIC TRAINING DATA
    # ------------------------------------------------------------

    for fold in (
        1,
        2,
    ):

        text_name = (
            f'fold{fold}-text.json'
        )

        gold_name = (
            f'fold{fold}-gold.json'
        )


        texts = load_text_map(
            paths[
                text_name
            ]
        )


        gold = load_gold_map(
            paths[
                gold_name
            ],
            texts,
            labels,
        )


        overlap = (
            set(train_text)
            & set(texts)
        )


        if overlap:

            raise ValueError(
                f'Training folds share IDs: '
                f'{sorted(overlap)[:8]}'
            )


        train_text.update(
            texts
        )

        train_gold.update(
            gold
        )


    # ------------------------------------------------------------
    # FOLD 3 = OFFICIAL HIDDEN-GOLD TEST DOCUMENTS
    # ------------------------------------------------------------

    test_text = load_text_map(
        paths[
            'fold3-text.json'
        ]
    )


    overlap = (
        set(test_text)
        & set(train_text)
    )


    if overlap:

        raise ValueError(
            f'Official Phase-2 test IDs overlap '
            f'public training IDs: '
            f'{sorted(overlap)[:10]}'
        )


    # ------------------------------------------------------------
    # DATASET FINGERPRINT
    # ------------------------------------------------------------

    source_hashes = {
        name:
        sha256_file(
            path
        )
        for name, path
        in paths.items()
    }


    fingerprint = hashlib.sha256(
        stable_json(
            source_hashes
        ).encode(
            'utf-8'
        )
    ).hexdigest()


    print()
    print(
        'Official Phase-2 dataset loaded successfully.'
    )

    print(
        f'  Training documents: {len(train_text)}'
    )

    print(
        f'  Test documents:     {len(test_text)}'
    )

    print(
        f'  Labels:             {len(labels)}'
    )

    print(
        f'  Dataset SHA256:     {fingerprint}'
    )

    print()


    return {
        'train_text':
            train_text,

        'train_gold':
            train_gold,

        'test_text':
            test_text,

        'labels':
            labels,

        'paths':
            paths,

        'fingerprint':
            fingerprint,
    }


# ================================================================
# TEST-ONLY LOADER USED BY inference_only MODE
# ================================================================

def load_phase2_test_only() -> dict[
    str,
    Any
]:

    label_path = discover_exact(
        'labels.txt'
    )

    test_path = discover_exact(
        'fold3-text.json'
    )


    labels = load_label_vocabulary(
        label_path
    )


    test_text = load_text_map(
        test_path
    )


    print()
    print(
        'Official Phase-2 test set loaded successfully.'
    )

    print(
        f'  Test documents: {len(test_text)}'
    )

    print(
        f'  Labels:         {len(labels)}'
    )

    print()


    return {
        'train_text':
            None,

        'train_gold':
            None,

        'test_text':
            test_text,

        'labels':
            labels,

        'paths': {
            'labels.txt':
                label_path,

            'fold3-text.json':
                test_path,
        },

        'fingerprint':
            None,

        'test_fingerprint':
            sha256_file(
                test_path
            ),
    }


# ================================================================
# DETERMINISTIC TRAIN/VALIDATION SPLIT
# ================================================================

def deterministic_split(
    ids: Sequence[str],
    fraction: float,
    seed: int,
) -> tuple[
    list[str],
    list[str],
]:

    if not (
        0 < fraction < 0.5
    ):

        raise ValueError(
            'Student-selected validation fraction '
            'must be in (0, 0.5)'
        )


    if len(ids) < 2:

        raise ValueError(
            'Need at least two documents '
            'for a train/validation split'
        )


    ordered = sorted(
        ids,
        key=lambda value:
        hashlib.sha256(
            f'{seed}:{value}'.encode(
                'utf-8'
            )
        ).hexdigest()
    )


    count = max(
        1,
        round(
            len(ordered)
            * fraction
        )
    )


    if count >= len(
        ordered
    ):

        raise ValueError(
            'Insufficient documents for '
            'held-out validation'
        )


    validation_ids = ordered[
        :count
    ]

    training_ids = ordered[
        count:
    ]


    return (
        training_ids,
        validation_ids,
    )


# ================================================================
# LOCAL STRICT MACRO P/R/F1
# ================================================================

def strict_macro_prf(
    predictions: Mapping[
        str,
        Sequence[
            Sequence[Any]
        ]
    ],
    gold: Mapping[
        str,
        Sequence[
            Sequence[Any]
        ]
    ],
    labels: Sequence[str],
) -> dict[str, Any]:

    if set(predictions) != set(
        gold
    ):

        raise ValueError(
            'Validation ID sets differ'
        )


    per_label = {}


    for label in labels:

        predicted = {
            (
                identifier,
                int(item[0]),
                int(item[1]),
            )

            for identifier, spans
            in predictions.items()

            for item
            in spans

            if item[2] == label
        }


        expected = {
            (
                identifier,
                int(item[0]),
                int(item[1]),
            )

            for identifier, spans
            in gold.items()

            for item
            in spans

            if item[2] == label
        }


        true_positive = len(
            predicted
            & expected
        )


        precision = (
            true_positive
            / len(predicted)
            if predicted
            else 0.0
        )


        recall = (
            true_positive
            / len(expected)
            if expected
            else 0.0
        )


        f1 = (
            2
            * precision
            * recall
            / (
                precision
                + recall
            )
            if (
                precision
                + recall
            )
            else 0.0
        )


        per_label[
            label
        ] = {
            'precision':
                precision,

            'recall':
                recall,

            'f1':
                f1,

            'tp':
                true_positive,

            'predicted':
                len(
                    predicted
                ),

            'gold':
                len(
                    expected
                ),
        }


    return {
        'strict_macro_f1_local':
            sum(
                item[
                    'f1'
                ]
                for item
                in per_label.values()
            )
            / len(
                labels
            ),

        'per_label':
            per_label,

        'note':
            (
                'Local strict-span approximation; '
                'compare with official evaluator'
            ),
    }


# ================================================================
# VALIDATE A COMPLETE PREDICTION MAP
# ================================================================

def validate_prediction_map(
    predictions: Mapping[
        str,
        Any
    ],
    texts: Mapping[
        str,
        str
    ],
    labels: Sequence[
        str
    ],
) -> dict[str, Any]:

    if not isinstance(
        predictions,
        dict
    ):

        raise TypeError(
            'Predictions must be an ID-keyed object'
        )


    expected_ids = set(
        texts
    )

    prediction_ids = set(
        predictions
    )


    if prediction_ids != expected_ids:

        missing = sorted(
            expected_ids
            - prediction_ids
        )

        extra = sorted(
            prediction_ids
            - expected_ids
        )

        raise ValueError(
            f'Prediction IDs differ: '
            f'missing={missing[:10]} '
            f'extra={extra[:10]}'
        )


    allowed = set(
        labels
    )


    total_spans = 0


    for identifier, spans in predictions.items():

        if not isinstance(
            spans,
            list
        ):

            raise TypeError(
                f'{identifier}: predictions '
                f'must be a list'
            )


        if any(
            not isinstance(
                item,
                (list, tuple)
            )
            for item
            in spans
        ):

            raise TypeError(
                f'{identifier}: each predicted '
                f'span must be an array'
            )


        canonical = [
            normalize_span(
                item,
                document_id=identifier,
                text=texts[
                    identifier
                ],
                allowed=allowed,
            )
            for item
            in spans
        ]


        if len(
            canonical
        ) != len(
            set(canonical)
        ):

            raise ValueError(
                f'{identifier}: duplicate '
                f'predicted spans'
            )


        total_spans += len(
            spans
        )


    return {
        'passed':
            True,

        'document_count':
            len(
                texts
            ),

        'span_count':
            total_spans,
    }


print(
    'Phase-2 dataset loader definitions: READY'
)

In [ ]:
def validate_experiment_plan() -> None:
    if len(EXPERIMENTS) != 6 or sorted(spec['slot'] for spec in EXPERIMENTS) != list(range(1, 7)):
        raise ValueError('Exactly six numbered experiment slots are required')
    if {spec['base_id'] for spec in EXPERIMENTS} != set(BASE_ARCHIVES):
        raise ValueError('The six submissions must involve all four Phase-1 architectures')
    methods = [spec['method'].lower() for spec in EXPERIMENTS]
    nondistill = [method for method in methods if method != 'knowledge_distillation']
    if len(set(nondistill)) < 5 or not any('lorma' in method for method in nondistill):
        raise ValueError('At least five distinct adaptation methods, including LORMA, are required')
    if methods.count('knowledge_distillation') < 1:
        raise ValueError('At least one knowledge-distillation experiment is required')
    for spec in EXPERIMENTS:
        if spec['base_id'] not in BASE_ARCHIVES or (spec['teacher_id'] is not None and spec['teacher_id'] not in BASE_ARCHIVES):
            raise ValueError(f'Unknown Phase-1 base/teacher: {spec}')
        if spec['method'] == 'knowledge_distillation' and spec['teacher_id'] in (None, spec['base_id']):
            raise ValueError('Distillation needs a distinct Phase-1 teacher')
        if Path(spec['module_filename']).name != spec['module_filename']:
            raise ValueError('Student method filename must be a basename')


def validate_training_config(slot: int) -> dict[str, Any]:
    config = dict(TRAINING_CONFIGS[slot])
    needed = ('max_steps', 'batch_size', 'gradient_accumulation_steps',
              'learning_rate', 'weight_decay', 'validation_interval',
              'max_tokens', 'window_stride', 'gradient_clip_norm')
    missing = [name for name in needed if config.get(name) is None]
    if missing:
        raise RuntimeError(f'Slot {slot}: student training configuration incomplete: {missing}')
    positive = ('max_steps', 'batch_size', 'gradient_accumulation_steps',
                'learning_rate', 'validation_interval', 'max_tokens', 'window_stride',
                'gradient_clip_norm',
                'checkpoint_interval', 'recovery_interval')
    if any(float(config[name]) <= 0 for name in positive) or float(config['weight_decay']) < 0:
        raise ValueError(f'Slot {slot}: invalid nonpositive scientific configuration')
    if config['optimizer'] != 'AdamW':
        raise ValueError('The supplied recovery runner supports AdamW only')
    return config


def student_module_path(spec: Mapping[str, Any]) -> Path:
    slot = int(spec['slot'])
    if slot in METHOD_SOURCE_OVERRIDES:
        path = Path(METHOD_SOURCE_OVERRIDES[slot]).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(set(path.resolve() for path in INPUT_ROOT.rglob(spec['module_filename']) if path.is_file()))
    recovered = experiment_directory(slot) / spec['module_filename']
    if not matches and recovered.is_file():
        return recovered
    if len(matches) != 1:
        raise RuntimeError(f'Slot {slot}: attach exactly one student-authored {spec["module_filename"]}; found {matches}')
    return matches[0]


def common_module_path() -> Path:
    filename = 'phase2_method_common.py'
    matches = sorted(set(path.resolve() for path in INPUT_ROOT.rglob(filename) if path.is_file()))
    recovered = WORK_ROOT / 'method_source' / filename
    if not matches and recovered.is_file():
        return recovered
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one {filename}; found {matches}')
    return matches[0]


def load_student_method(spec: Mapping[str, Any], destination: Path):
    source = student_module_path(spec)
    data = source.read_bytes()
    ast.parse(data.decode('utf-8'), filename=str(source))
    destination.parent.mkdir(parents=True, exist_ok=True)
    shared_source = common_module_path()
    shared_data = shared_source.read_bytes()
    ast.parse(shared_data.decode('utf-8'), filename=str(shared_source))
    shared_destination = WORK_ROOT / 'method_source' / shared_source.name
    shared_destination.parent.mkdir(parents=True, exist_ok=True)
    if shared_destination.exists() and shared_destination.read_bytes() != shared_data:
        raise RuntimeError('Shared method source changed within this run workspace')
    if not shared_destination.exists():
        shared_destination.write_bytes(shared_data)
    local_common = destination.parent / shared_source.name
    if local_common.exists() and local_common.read_bytes() != shared_data:
        raise RuntimeError('Shared method source changed within experiment workspace')
    if not local_common.exists():
        local_common.write_bytes(shared_data)
    if destination.exists() and destination.read_bytes() != data:
        raise RuntimeError(f'Slot {spec["slot"]}: student method changed within an experiment workspace')
    if not destination.exists():
        with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, destination)
    if str(destination.parent) not in sys.path:
        sys.path.insert(0, str(destination.parent))
    # A common module with the same name must not retain a prior workspace version.
    sys.modules.pop('phase2_method_common', None)
    module = import_local_source(destination, f'phase2_method_{spec["slot"]}_{sha256_file(destination)[:12]}')
    for name in ('build_task_model', 'collate_batch', 'predict_spans'):
        if not callable(getattr(module, name, None)):
            raise AttributeError(f'Slot {spec["slot"]}: student module lacks {name}')
    combined = hashlib.sha256(data + b'\0' + shared_data).hexdigest()
    return module, combined


def choose_gpu_runtime() -> dict[str, Any]:
    if not torch.cuda.is_available():
        raise RuntimeError('Phase-2 training requires a Kaggle GPU; enable the accelerator')
    ids = list(range(torch.cuda.device_count()))
    bf16 = all(torch.cuda.get_device_capability(i)[0] >= 8 for i in ids) and torch.cuda.is_bf16_supported()
    return {'ids': ids, 'primary': 'cuda:0', 'precision': 'bf16' if bf16 else 'fp16',
            'amp_dtype': torch.bfloat16 if bf16 else torch.float16,
            'scaler_enabled': not bf16,
            'names': [torch.cuda.get_device_name(i) for i in ids]}


def move_batch(value: Any, device: str) -> Any:
    if torch.is_tensor(value):
        return value.to(device, non_blocking=True)
    if isinstance(value, dict):
        return {key: move_batch(item, device) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(move_batch(item, device) for item in value)
    return value


def task_loss(output: Any) -> torch.Tensor:
    loss = output['loss'] if isinstance(output, Mapping) else output
    if not torch.is_tensor(loss) or not torch.isfinite(loss).all():
        raise FloatingPointError('Student task must return a finite torch loss tensor')
    return loss.mean()


def make_optimizer(model: torch.nn.Module, config: Mapping[str, Any]):
    params = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not params:
        raise RuntimeError('Student task has no trainable parameters')
    kwargs = {'lr': float(config['learning_rate']), 'weight_decay': float(config['weight_decay'])}
    try:
        return torch.optim.AdamW(params, fused=True, **kwargs), 'fused'
    except (TypeError, RuntimeError):
        return torch.optim.AdamW(params, **kwargs), 'standard'


def make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


In [ ]:
class DocumentRows(Dataset):
    def __init__(self, texts: Mapping[str, str], gold: Mapping[str, Any], ids: Sequence[str]):
        self.rows = [{'id': identifier, 'text': texts[identifier], 'spans': gold[identifier]}
                     for identifier in ids]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index: int):
        return self.rows[index]


class ResumableBatchSampler(Sampler[list[int]]):
    def __init__(self, length: int, batch_size: int, seed: int):
        if length <= 0 or batch_size <= 0:
            raise ValueError('Sampler needs documents and a positive batch size')
        self.length, self.batch_size, self.seed = int(length), int(batch_size), int(seed)
        self.epoch, self.cursor = 0, 0

    def _order(self) -> list[int]:
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        return torch.randperm(self.length, generator=generator).tolist()

    def __iter__(self):
        order = self._order()
        while self.cursor < self.length:
            end = min(self.length, self.cursor + self.batch_size)
            indices = order[self.cursor:end]
            self.cursor = end
            yield indices

    def __len__(self):
        return math.ceil((self.length - self.cursor) / self.batch_size)

    def next_epoch(self):
        self.epoch += 1
        self.cursor = 0

    def state_dict(self):
        return {'length': self.length, 'batch_size': self.batch_size, 'seed': self.seed,
                'epoch': self.epoch, 'cursor': self.cursor}

    def load_state_dict(self, state):
        expected = {'length': self.length, 'batch_size': self.batch_size, 'seed': self.seed}
        if {key: state[key] for key in expected} != expected:
            raise RuntimeError('Sampler data, batch size, or seed changed')
        self.epoch, self.cursor = int(state['epoch']), int(state['cursor'])
        if not 0 <= self.cursor <= self.length:
            raise ValueError('Invalid sampler cursor')


def capture_rng():
    np_state = np.random.get_state()
    return {'python': random.getstate(), 'numpy': (np_state[0], np_state[1].tolist(),
            np_state[2], np_state[3], np_state[4]),
            'torch_cpu': torch.get_rng_state(),
            'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def restore_rng(state):
    random.setstate(state['python'])
    name, keys, position, gaussian_flag, cached = state['numpy']
    np.random.set_state((name, np.array(keys, dtype=np.uint32), position, gaussian_flag, cached))
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and state['torch_cuda']:
        if len(state['torch_cuda']) != torch.cuda.device_count():
            raise RuntimeError('CUDA device count differs from saved RNG state')
        torch.cuda.set_rng_state_all(state['torch_cuda'])


def bare_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def atomic_torch_save(path: Path, state: Mapping[str, Any]) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.pt', delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(dict(state), temporary)
        check = torch.load(temporary, map_location='cpu', weights_only=True)
        if check.get('format') != state.get('format') or check.get('step') != state.get('step'):
            raise RuntimeError(f'Checkpoint readback failed: {path}')
        if path.exists():
            previous = path.with_name(path.stem + '.previous_known_good.pt')
            with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.pt', delete=False) as handle:
                prior_copy = Path(handle.name)
            prior_copy.unlink()
            try:
                try:
                    os.link(path, prior_copy)
                except OSError:
                    shutil.copyfile(path, prior_copy)
                os.replace(prior_copy, previous)
            finally:
                prior_copy.unlink(missing_ok=True)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return sha256_file(path)


def experiment_directory(slot: int) -> Path:
    return WORK_ROOT / 'experiments' / f'submission_{slot:02d}'


def experiment_identity(spec, config, base_meta, dataset_fingerprint, method_sha):
    content = {'spec': dict(spec), 'config': dict(config), 'base': base_meta,
               'dataset_fingerprint': dataset_fingerprint, 'method_sha256': method_sha}
    return hashlib.sha256(stable_json(content).encode()).hexdigest()


def checkpoint_payload(model, optimizer, scheduler, scaler, sampler, *, step, best_score,
                       identity, history_path, config, base_meta, method_sha, dataset_fingerprint):
    return {
        'format': 'cs781_phase2_training_v1', 'step': int(step),
        'best_validation_score': best_score, 'identity': identity,
        'model_state': bare_model(model).state_dict(),
        'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'sampler_state': sampler.state_dict(), 'rng_state': capture_rng(),
        'training_config': dict(config), 'base_meta': base_meta,
        'method_sha256': method_sha, 'dataset_fingerprint': dataset_fingerprint,
        'history_committed_step': int(step),
        'history_sha256': sha256_file(history_path) if history_path.exists() else None,
    }


def restore_training_checkpoint(path, model, optimizer, scheduler, scaler, sampler,
                                identity, config, method_sha, dataset_fingerprint):
    state = torch.load(path, map_location='cpu', weights_only=True)
    if state.get('format') != 'cs781_phase2_training_v1' or state.get('identity') != identity:
        raise RuntimeError('Checkpoint experiment identity mismatch')
    if state.get('training_config') != dict(config) or state.get('method_sha256') != method_sha:
        raise RuntimeError('Checkpoint training/method configuration changed')
    if state.get('dataset_fingerprint') != dataset_fingerprint:
        raise RuntimeError('Checkpoint dataset changed')
    bare_model(model).load_state_dict(state['model_state'], strict=True)
    optimizer.load_state_dict(state['optimizer_state'])
    scheduler.load_state_dict(state['scheduler_state'])
    if scaler is not None:
        if state.get('scaler_state') is None:
            raise RuntimeError('FP16 scaler state missing from checkpoint')
        scaler.load_state_dict(state['scaler_state'])
    elif state.get('scaler_state') is not None:
        raise RuntimeError('Checkpoint precision policy changed')
    sampler.load_state_dict(state['sampler_state'])
    return state


def reconcile_history(path: Path, committed_step: int) -> None:
    if not path.exists():
        if committed_step:
            raise RuntimeError('Checkpoint exists but training history is missing')
        return
    lines = path.read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    committed = [row for row in rows if int(row['step']) <= committed_step]
    if committed and int(committed[-1]['step']) != committed_step:
        raise RuntimeError('History does not contain the checkpoint step')
    excess = [row for row in rows if int(row['step']) > committed_step]
    if excess:
        archive = path.with_name(f'{path.stem}.uncommitted.after_{committed_step}.{int(time.time())}.jsonl')
        shutil.copy2(path, archive)
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent,
                                         suffix='.jsonl', delete=False) as handle:
            temporary = Path(handle.name)
            for row in committed:
                handle.write(stable_json(row) + '\n')
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)


def recovery_name(slot: int) -> str:
    return f'phase2_submission{slot:02d}_recovery.zip'


def export_recovery(slot: int) -> Path:
    directory = experiment_directory(slot)
    output = directory / recovery_name(slot)
    items = sorted(path for path in directory.rglob('*') if path.is_file()
                   and path != output and path.name != 'recovery_receipt.json'
                   and not path.name.endswith('.previous_known_good.pt')
                   and not path.name.endswith('.tmp'))
    manifest = {'slot': slot, 'files': [
        {'path': path.relative_to(directory).as_posix(), 'bytes': path.stat().st_size,
         'sha256': sha256_file(path)} for path in items
    ]}
    with tempfile.NamedTemporaryFile(dir=directory, suffix='.zip', delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, 'w', allowZip64=True) as archive:
            for row in manifest['files']:
                name = row['path']
                compression = zipfile.ZIP_STORED if name.endswith('.pt') else zipfile.ZIP_DEFLATED
                archive.write(directory / name, name, compress_type=compression)
            archive.writestr('recovery_integrity.json', stable_json(manifest))
        verify_recovery_zip(temporary, slot)
        os.replace(temporary, output)
    finally:
        temporary.unlink(missing_ok=True)
    atomic_json(directory / 'recovery_receipt.json',
                {'path': str(output), 'sha256': sha256_file(output), 'bytes': output.stat().st_size})
    return output


def verify_recovery_zip(path: Path, slot: int):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or 'recovery_integrity.json' not in names:
            raise ValueError('Recovery ZIP lacks an unambiguous manifest')
        manifest = json.loads(archive.read('recovery_integrity.json'))
        rows = {row['path']: row for row in manifest['files']}
        if manifest['slot'] != slot or len(rows) != len(manifest['files']) or set(rows) != set(names) - {'recovery_integrity.json'}:
            raise ValueError('Recovery ZIP manifest/member mismatch')
        for name, row in rows.items():
            if not _safe_archive_member(name) or archive.getinfo(name).is_dir():
                raise ValueError(f'Unsafe recovery member: {name}')
            digest = hashlib.sha256()
            with archive.open(name) as stream:
                for chunk in iter(lambda: stream.read(1 << 20), b''):
                    digest.update(chunk)
            if digest.hexdigest() != row['sha256'] or archive.getinfo(name).file_size != row['bytes']:
                raise ValueError(f'Recovery member integrity failed: {name}')
    return manifest


def restore_attached_recovery(slot: int) -> Path | None:
    if slot in RESTORED_RECOVERY_SLOTS:
        return RESTORED_RECOVERY_SLOTS[slot]
    name = recovery_name(slot)
    attached = discover_exact(name, required=False)
    if attached is None:
        return None
    manifest = verify_recovery_zip(attached, slot)
    directory = experiment_directory(slot).resolve()
    directory.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(attached) as archive:
        for row in manifest['files']:
            relative = row['path']
            target = (directory / relative).resolve()
            if directory not in target.parents:
                raise ValueError(f'Recovery member escapes experiment directory: {relative}')
            extract_member_verified(archive, relative, target,
                                    {relative: row})
    RESTORED_RECOVERY_SLOTS[slot] = attached
    return attached


In [ ]:
def build_task(method, base_id, spec, config, labels, device):
    base_model, tokenizer, base_meta = load_phase1_base(base_id, 'cpu')
    teacher = None
    teacher_tokenizer = None
    teacher_meta = None
    if spec['teacher_id'] is not None:
        teacher, teacher_tokenizer, teacher_meta = load_phase1_base(spec['teacher_id'], 'cpu')
        if teacher_meta['parameter_count'] <= base_meta['parameter_count']:
            raise ValueError('The declared Phase-1 distillation teacher must be larger than the student base')
        teacher_spec = EXPERIMENTS[0]
        if teacher_spec['base_id'] != spec['teacher_id'] or teacher_spec['method'] != 'full_finetune':
            raise RuntimeError('Distillation requires the completed slot-1 NER teacher')
        teacher_directory = experiment_directory(1)
        teacher_method, teacher_method_sha = load_student_method(
            teacher_spec, teacher_directory / teacher_spec['module_filename'])
        teacher_config = validate_training_config(1)
        teacher_selected_path = teacher_directory / 'selected_model.pt'
        teacher_result_path = teacher_directory / 'training_result.json'
        if not teacher_selected_path.is_file() or not teacher_result_path.is_file():
            raise RuntimeError('Complete slot 1 before distillation; trained NER teacher missing')
        teacher_result = load_json(teacher_result_path)
        if teacher_result.get('status') != 'complete':
            raise RuntimeError('Slot-1 NER teacher training is incomplete')
        teacher_selected = torch.load(teacher_selected_path, map_location='cpu', weights_only=True)
        if (teacher_selected.get('method_sha256') != teacher_method_sha or
                teacher_selected.get('training_config') != teacher_config):
            raise RuntimeError('Slot-1 NER teacher checkpoint/source/config mismatch')
        teacher = teacher_method.build_task_model(teacher, teacher_tokenizer, labels, teacher_config)
        teacher.load_state_dict(teacher_selected['model_state'], strict=True)
        teacher.eval()
        for parameter in teacher.parameters():
            parameter.requires_grad_(False)
        teacher_meta = {**teacher_meta, 'ner_selected_sha256': sha256_file(teacher_selected_path),
                        'ner_method_sha256': teacher_method_sha,
                        'ner_validation_macro_f1': teacher_selected['strict_macro_f1_local']}
    task = method.build_task_model(base_model, tokenizer, labels, dict(config),
                                   teacher_model=teacher, teacher_tokenizer=teacher_tokenizer)
    if not isinstance(task, torch.nn.Module):
        raise TypeError('Student build_task_model must return a torch.nn.Module')
    task = task.to(device)
    if torch.cuda.device_count() > 1:
        if int(config['batch_size']) < torch.cuda.device_count() or int(config['batch_size']) % torch.cuda.device_count():
            raise ValueError('Batch size must be divisible by the active GPU count')
        task = torch.nn.DataParallel(task, device_ids=list(range(torch.cuda.device_count())))
    return task, tokenizer, teacher_tokenizer, base_meta, teacher_meta


@torch.inference_mode()
def evaluate_student_method(method, task, tokenizer, data, val_ids, config, device):
    task.eval()
    predictions = {}
    for identifier in val_ids:
        spans = method.predict_spans(task, data['train_text'][identifier],
                                     tokenizer, data['labels'], dict(config))
        if not isinstance(spans, list):
            raise TypeError(f'Validation prediction for {identifier} is not a list')
        predictions[identifier] = spans
    validate_prediction_map(predictions, {identifier: data['train_text'][identifier] for identifier in val_ids},
                            data['labels'])
    gold = {identifier: data['train_gold'][identifier] for identifier in val_ids}
    report = strict_macro_prf(predictions, gold, data['labels'])
    task.train()
    return report


def make_scheduler(method, optimizer, config):
    custom = getattr(method, 'build_scheduler', None)
    scheduler = custom(optimizer, dict(config)) if callable(custom) else torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda _step: 1.0
    )
    if not hasattr(scheduler, 'state_dict') or not hasattr(scheduler, 'load_state_dict'):
        raise TypeError('Student scheduler must support state_dict and load_state_dict')
    return scheduler


def create_loader(method, tokenizer, teacher_tokenizer, labels, config, texts, gold, ids, sampler):
    dataset = DocumentRows(texts, gold, ids)
    return DataLoader(dataset, batch_sampler=sampler, num_workers=0,
                      pin_memory=True,
                      collate_fn=lambda rows: method.collate_batch(
                          rows, tokenizer, labels, dict(config), teacher_tokenizer=teacher_tokenizer))


def run_gpu_smoke(spec, method, data, train_ids, val_ids, config, directory, method_sha, identity):
    manifest = directory / 'gpu_smoke.json'
    if manifest.exists():
        report = load_json(manifest)
        if report.get('identity') != identity or report.get('status') != 'PASS':
            raise RuntimeError('Stale or failed GPU smoke record')
        return report
    gpu = choose_gpu_runtime()
    task, tokenizer, teacher_tokenizer, base_meta, _teacher_meta = build_task(
        method, spec['base_id'], spec, config, data['labels'], gpu['primary']
    )
    sampler = ResumableBatchSampler(len(train_ids), int(config['batch_size']), int(config['seed']))
    loader = create_loader(method, tokenizer, teacher_tokenizer, data['labels'], config,
                           data['train_text'], data['train_gold'], train_ids, sampler)
    started = time.monotonic()
    batch = move_batch(next(iter(loader)), gpu['primary'])
    batch_prep_seconds = time.monotonic() - started
    optimizer, backend = make_optimizer(task, config)
    scheduler = make_scheduler(method, optimizer, config)
    scaler = make_scaler(gpu['scaler_enabled'])
    for gpu_id in gpu['ids']:
        torch.cuda.reset_peak_memory_stats(gpu_id)
        torch.cuda.synchronize(gpu_id)
    started = time.monotonic()
    task.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=gpu['amp_dtype']):
        loss = task_loss(task(**batch))
    (scaler.scale(loss) if gpu['scaler_enabled'] else loss).backward()
    if gpu['scaler_enabled']:
        scaler.unscale_(optimizer)
    gradient_norm = torch.nn.utils.clip_grad_norm_(task.parameters(), float(config['gradient_clip_norm']))
    if not math.isfinite(float(gradient_norm)):
        raise FloatingPointError('GPU smoke produced nonfinite gradient norm')
    if gpu['scaler_enabled']:
        before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() < before:
            raise FloatingPointError('GPU smoke FP16 optimizer step overflowed')
    else:
        optimizer.step()
    scheduler.step()
    for gpu_id in gpu['ids']:
        torch.cuda.synchronize(gpu_id)
    compute_seconds = time.monotonic() - started
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name) / 'gpu_smoke.pt'
        payload = checkpoint_payload(task, optimizer, scheduler, scaler if gpu['scaler_enabled'] else None,
                                     sampler, step=1, best_score=None, identity=identity,
                                     history_path=directory / 'not_yet_created_history.jsonl',
                                     config=config, base_meta=base_meta, method_sha=method_sha,
                                     dataset_fingerprint=data['fingerprint'])
        atomic_torch_save(temporary, payload)
        restore_training_checkpoint(temporary, task, optimizer, scheduler,
                                    scaler if gpu['scaler_enabled'] else None, sampler,
                                    identity, config, method_sha, data['fingerprint'])
        if not optimizer.state:
            raise AssertionError('GPU smoke did not restore optimizer state')
    probe_id = val_ids[0]
    probe_text = data['train_text'][probe_id]
    encoding = tokenizer.backend.encode(probe_text, add_special_tokens=False)
    if len(encoding.ids) != len(encoding.offsets) or any(
        not 0 <= start <= end <= len(probe_text) for start, end in encoding.offsets
    ):
        raise AssertionError('BPE offset map is invalid for a held-out legal document')
    task.eval()
    with torch.inference_mode():
        probe_spans = method.predict_spans(task, probe_text,
                                           tokenizer, data['labels'], dict(config))
    validate_prediction_map({probe_id: probe_spans}, {probe_id: probe_text}, data['labels'])
    report = {'status': 'PASS', 'identity': identity, 'loss': float(loss.detach().cpu()),
              'gradient_norm': float(gradient_norm), 'batch_prep_seconds': batch_prep_seconds,
              'single_microbatch_compute_seconds': compute_seconds,
              'precision': gpu['precision'], 'gpu_names': gpu['names'],
              'peak_memory_gib': {str(i): torch.cuda.max_memory_allocated(i) / 2**30 for i in gpu['ids']},
              'optimizer_backend': backend, 'checkpoint_restore': 'PASS',
              'heldout_prediction_contract': 'PASS',
              'timing_note': 'One microbatch, not a full accumulated optimizer step'}
    atomic_json(manifest, report)
    del task, optimizer, scheduler, scaler, batch
    torch.cuda.empty_cache()
    return report


def save_selected_model(directory, task, *, step, score, identity, config, method_sha,
                        base_meta, tokenizer_fingerprint, dataset_fingerprint):
    path = directory / 'selected_model.pt'
    state = {'format': 'cs781_phase2_selected_v1', 'step': int(step),
             'strict_macro_f1_local': float(score), 'identity': identity,
             'model_state': bare_model(task).state_dict(), 'training_config': dict(config),
             'method_sha256': method_sha, 'base_meta': base_meta,
             'tokenizer_fingerprint': tokenizer_fingerprint,
             'dataset_fingerprint': dataset_fingerprint}
    digest = atomic_torch_save(path, state)
    return {'path': str(path), 'sha256': digest, 'step': step, 'score': score}


def train_one_experiment(spec, data, train_ids, val_ids, session_deadline):
    slot = int(spec['slot'])
    directory = experiment_directory(slot)
    directory.mkdir(parents=True, exist_ok=True)
    attached_recovery = restore_attached_recovery(slot)
    config = validate_training_config(slot)
    method, method_sha = load_student_method(spec, directory / spec['module_filename'])
    gpu = choose_gpu_runtime()
    if gpu['precision'] == 'fp16' and not gpu['scaler_enabled']:
        raise RuntimeError('FP16 must have gradient scaling')
    task, tokenizer, teacher_tokenizer, base_meta, teacher_meta = build_task(
        method, spec['base_id'], spec, config, data['labels'], gpu['primary']
    )
    identity = experiment_identity(spec, config,
                                   {'student': base_meta, 'teacher': teacher_meta},
                                   data['fingerprint'], method_sha)
    identity_path = directory / 'experiment_identity.json'
    identity_record = {'identity': identity, 'spec': dict(spec), 'config': config,
                       'base_meta': base_meta, 'teacher_meta': teacher_meta,
                       'dataset_fingerprint': data['fingerprint'], 'method_sha256': method_sha}
    if identity_path.exists() and load_json(identity_path) != identity_record:
        raise RuntimeError(f'Slot {slot}: experiment identity changed after initialization')
    atomic_json(identity_path, identity_record)
    if attached_recovery is not None:
        print(f'Slot {slot}: restored verified recovery from {attached_recovery}')
    latest = directory / 'checkpoint.latest.pt'
    history = directory / 'training.jsonl'
    sampler = ResumableBatchSampler(len(train_ids), int(config['batch_size']), int(config['seed']))
    loader = create_loader(method, tokenizer, teacher_tokenizer, data['labels'], config,
                           data['train_text'], data['train_gold'], train_ids, sampler)
    optimizer, backend = make_optimizer(task, config)
    scheduler = make_scheduler(method, optimizer, config)
    scaler = make_scaler(gpu['scaler_enabled'])
    best_score, step = -math.inf, 0
    if latest.exists():
        if RUN_MODE == 'fresh':
            raise RuntimeError(f'Slot {slot}: fresh mode found a compatible checkpoint; choose resume')
        state = restore_training_checkpoint(
            latest, task, optimizer, scheduler, scaler if gpu['scaler_enabled'] else None,
            sampler, identity, config, method_sha, data['fingerprint']
        )
        step = int(state['step'])
        best_score = float(state['best_validation_score']) if state['best_validation_score'] is not None else -math.inf
        reconcile_history(history, step)
        print(f'Slot {slot}: resuming optimizer step {step}')
    else:
        if (RUN_MODE == 'resume' and slot in (RESUME_EXPECTED_SLOTS or set())) or history.exists() or (directory / 'selected_model.pt').exists():
            raise RuntimeError(f'Slot {slot}: no latest checkpoint; refusing a silent step-zero restart')
        del task, optimizer, scheduler, scaler, loader
        torch.cuda.empty_cache()
        smoke = run_gpu_smoke(spec, method, data, train_ids, val_ids, config,
                              directory, method_sha, identity)
        print(f'Slot {slot} GPU smoke:', smoke)
        task, tokenizer, teacher_tokenizer, base_meta, teacher_meta = build_task(
            method, spec['base_id'], spec, config, data['labels'], gpu['primary']
        )
        sampler = ResumableBatchSampler(len(train_ids), int(config['batch_size']), int(config['seed']))
        loader = create_loader(method, tokenizer, teacher_tokenizer, data['labels'], config,
                               data['train_text'], data['train_gold'], train_ids, sampler)
        optimizer, backend = make_optimizer(task, config)
        scheduler = make_scheduler(method, optimizer, config)
        scaler = make_scaler(gpu['scaler_enabled'])
    iterator = iter(loader)
    if latest.exists() and step:
        restore_rng(state['rng_state'])
    if step >= int(config['max_steps']):
        if not (directory / 'selected_model.pt').exists():
            raise RuntimeError(f'Slot {slot}: training marked complete but selected checkpoint is absent')
        completed = {'slot': slot, 'status': 'complete', 'step': step,
                     'target_step': int(config['max_steps']),
                     'selected': str(directory / 'selected_model.pt')}
        atomic_json(directory / 'training_result.json', completed)
        export_recovery(slot)
        return completed
    if time.monotonic() >= session_deadline:
        if latest.exists():
            export_recovery(slot)
        return {'slot': slot, 'status': 'time_budget', 'step': step}
    started = time.monotonic()
    try:
        while step < int(config['max_steps']):
            if time.monotonic() >= session_deadline:
                break
            step_started = time.monotonic()
            task.train()
            optimizer.zero_grad(set_to_none=True)
            loss_total = 0.0
            for _ in range(int(config['gradient_accumulation_steps'])):
                try:
                    batch = next(iterator)
                except StopIteration:
                    sampler.next_epoch()
                    iterator = iter(loader)
                    batch = next(iterator)
                batch = move_batch(batch, gpu['primary'])
                with torch.autocast(device_type='cuda', dtype=gpu['amp_dtype']):
                    loss = task_loss(task(**batch))
                    backward_loss = loss / int(config['gradient_accumulation_steps'])
                (scaler.scale(backward_loss) if gpu['scaler_enabled'] else backward_loss).backward()
                loss_total += float(loss.detach().cpu())
            if gpu['scaler_enabled']:
                scaler.unscale_(optimizer)
            norm = float(torch.nn.utils.clip_grad_norm_(task.parameters(),
                                                       float(config['gradient_clip_norm'])).cpu())
            if not math.isfinite(norm):
                raise FloatingPointError(f'Slot {slot}: nonfinite gradient norm')
            if gpu['scaler_enabled']:
                before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() < before:
                    append_jsonl(directory / 'overflow.jsonl',
                                 {'attempted_step': step + 1, 'epoch': sampler.epoch})
                    continue
            else:
                optimizer.step()
            scheduler.step()
            step += 1
            validation = None
            improved = False
            if step % int(config['validation_interval']) == 0 or step == int(config['max_steps']):
                cap = int(config.get('validation_document_cap', len(val_ids)))
                validation_ids = val_ids if step == int(config['max_steps']) else val_ids[:cap]
                validation = evaluate_student_method(method, task, tokenizer, data, validation_ids,
                                                     config, gpu['primary'])
                validation['evaluated_documents'] = len(validation_ids)
                validation['full_validation'] = len(validation_ids) == len(val_ids)
                atomic_json(directory / f'validation_step_{step:07d}.json', validation)
                score = float(validation['strict_macro_f1_local'])
                improved = score > best_score
                best_score = max(best_score, score)
            for gpu_id in gpu['ids']:
                torch.cuda.synchronize(gpu_id)
            optimizer_step_seconds = time.monotonic() - step_started
            row = {'step': step, 'epoch': sampler.epoch, 'loss': loss_total / int(config['gradient_accumulation_steps']),
                   'validation_macro_f1': None if validation is None else validation['strict_macro_f1_local'],
                   'learning_rate': optimizer.param_groups[0]['lr'], 'gradient_norm': norm,
                   'seconds_elapsed': time.monotonic() - started,
                   'optimizer_step_seconds': optimizer_step_seconds,
                   'nominal_documents_per_second': (
                       int(config['batch_size']) * int(config['gradient_accumulation_steps']) /
                       max(optimizer_step_seconds, 1e-9)
                   ),
                   'gpu_memory_gib': {str(i): torch.cuda.max_memory_allocated(i) / 2**30 for i in gpu['ids']},
                   'optimizer_backend': backend, 'precision': gpu['precision']}
            append_jsonl(history, row)
            if step % int(config['checkpoint_interval']) == 0 or improved or step == int(config['max_steps']):
                payload = checkpoint_payload(task, optimizer, scheduler,
                                             scaler if gpu['scaler_enabled'] else None, sampler,
                                             step=step, best_score=best_score, identity=identity,
                                             history_path=history, config=config, base_meta=base_meta,
                                             method_sha=method_sha, dataset_fingerprint=data['fingerprint'])
                atomic_torch_save(latest, payload)
                if improved:
                    save_selected_model(directory, task, step=step, score=best_score,
                                        identity=identity, config=config, method_sha=method_sha,
                                        base_meta=base_meta,
                                        tokenizer_fingerprint=base_meta['tokenizer_fingerprint'],
                                        dataset_fingerprint=data['fingerprint'])
                if step % int(config['recovery_interval']) == 0:
                    export_recovery(slot)
        if not latest.exists() or int(torch.load(latest, map_location='cpu', weights_only=True)['step']) != step:
            payload = checkpoint_payload(task, optimizer, scheduler,
                                         scaler if gpu['scaler_enabled'] else None, sampler,
                                         step=step, best_score=best_score, identity=identity,
                                         history_path=history, config=config, base_meta=base_meta,
                                         method_sha=method_sha, dataset_fingerprint=data['fingerprint'])
            atomic_torch_save(latest, payload)
        result = {'slot': slot, 'status': 'complete' if step == int(config['max_steps']) else 'time_budget',
                  'step': step, 'target_step': int(config['max_steps']),
                  'best_strict_macro_f1_local': None if best_score == -math.inf else best_score,
                  'checkpoint': str(latest), 'recovery': str(directory / recovery_name(slot)),
                  'selected': str(directory / 'selected_model.pt')}
        atomic_json(directory / 'training_result.json', result)
        export_recovery(slot)
        return result
    except Exception as exc:
        append_jsonl(directory / 'failure_events.jsonl',
                     {'last_committed_step': step, 'error': f'{type(exc).__name__}: {exc}'})
        if latest.exists():
            export_recovery(slot)
        raise


In [ ]:
# ================================================================
# FINAL PHASE-2 EMPTY-DOCUMENT FIX
#
# Why this is needed:
# Official fold3-text.json contains at least one legitimate empty
# document, e.g. ID 1305957 -> "".
#
# Correct behavior:
#   - accept the empty document in the official test set
#   - produce [] predictions for it
#   - DO NOT call the model/tokenizer on an empty string
#
# Paste this entire cell immediately BEFORE:
# "Resumable Phase-2 test predictions and six validated ZIP packages"
# ================================================================


def _extract_document_text(
    identifier: str,
    value: Any,
    *,
    path: Path,
) -> str | None:
    """
    Robust extraction of one document from the official text JSON.

    Empty strings are VALID because the official Phase-2 test set
    contains at least one empty document.

    Supported forms:
        "id": "text"

    and:
        "id": {"text": "text"}

    Metadata-like top-level entries may be ignored.
    """

    # ------------------------------------------------------------
    # DIRECT STRING
    #
    # IMPORTANT:
    # Empty string is intentionally accepted.
    # ------------------------------------------------------------
    if isinstance(value, str):
        return value


    # ------------------------------------------------------------
    # WRAPPED TEXT OBJECT
    # ------------------------------------------------------------
    if isinstance(value, dict):

        preferred_keys = (
            'text',
            'document',
            'document_text',
            'content',
            'body',
            'input_text',
        )

        for key in preferred_keys:

            if key in value:

                candidate = value[key]

                if isinstance(candidate, str):
                    return candidate


        # Defensive fallback:
        # accept exactly one string-valued field.
        string_values = [
            candidate
            for candidate in value.values()
            if isinstance(candidate, str)
        ]

        if len(string_values) == 1:
            return string_values[0]


    # ------------------------------------------------------------
    # OPTIONAL METADATA ENTRIES
    # ------------------------------------------------------------
    normalized_id = str(identifier).strip().lower()

    metadata_names = {
        'metadata',
        '_metadata',
        '__metadata__',
        'meta',
        '_meta',
        '__meta__',
        'info',
        '_info',
        '__info__',
    }

    if (
        normalized_id in metadata_names
        or normalized_id.startswith('__')
    ):
        print(
            f'Ignoring metadata entry '
            f'{identifier!r} in {path.name}'
        )
        return None


    # ------------------------------------------------------------
    # ANYTHING ELSE IS UNSAFE TO GUESS
    # ------------------------------------------------------------
    preview = repr(value)

    if len(preview) > 500:
        preview = preview[:500] + ' ...'

    raise ValueError(
        f'Unsupported text representation in {path}\n'
        f'Document ID: {identifier!r}\n'
        f'Value type: {type(value).__name__}\n'
        f'Value preview: {preview}'
    )


def load_text_map(
    path: Path
) -> dict[str, str]:
    """
    Load official Phase-2 text files.

    Empty strings are retained exactly as provided because character
    offsets depend on the original source text.
    """

    path = Path(path)

    raw = load_json(path)

    if not isinstance(raw, dict):
        raise ValueError(
            f'Expected ID-to-text JSON object: {path}'
        )

    if not raw:
        raise ValueError(
            f'Text JSON object is empty: {path}'
        )


    result: dict[str, str] = {}

    direct_strings = 0
    wrapped_strings = 0
    empty_documents = 0
    ignored_metadata = 0


    for raw_identifier, raw_value in raw.items():

        identifier = str(raw_identifier)

        was_direct = isinstance(
            raw_value,
            str
        )

        text = _extract_document_text(
            identifier,
            raw_value,
            path=path,
        )


        if text is None:
            ignored_metadata += 1
            continue


        if not isinstance(text, str):
            raise TypeError(
                f'{identifier}: normalized document '
                f'is not a string'
            )


        if identifier in result:
            raise ValueError(
                f'Duplicate document ID '
                f'{identifier!r} in {path}'
            )


        # --------------------------------------------------------
        # EMPTY STRING IS VALID.
        # Do NOT discard it and do NOT replace it.
        # --------------------------------------------------------
        if text == '':
            empty_documents += 1


        result[identifier] = text


        if was_direct:
            direct_strings += 1
        else:
            wrapped_strings += 1


    if not result:
        raise ValueError(
            f'No documents found in {path}'
        )


    print(
        f'Loaded {len(result)} documents from {path.name} '
        f'({direct_strings} direct strings, '
        f'{wrapped_strings} wrapped texts, '
        f'{empty_documents} empty documents'
        + (
            f', {ignored_metadata} metadata entries ignored'
            if ignored_metadata
            else ''
        )
        + ')'
    )


    return result


# ================================================================
# REPLACE TEST-PREDICTION FUNCTION
# ================================================================

def generate_test_predictions(
    spec,
    data,
    config,
    method,
    method_sha,
    *,
    retry_rounds=3,
    session_deadline=None,
):
    """
    Generate Phase-2 test predictions.

    Empty official documents are assigned [] directly instead of being
    passed to the model/tokenizer.
    """

    slot = int(spec['slot'])

    directory = experiment_directory(
        slot
    )


    # ============================================================
    # LOAD SELECTED TRAINED TASK MODEL
    # ============================================================

    model, tokenizer, selected_path, selected_state = (
        load_selected_task(
            spec,
            data,
            config,
            method,
            method_sha,
            'cuda:0',
        )
    )


    partial_path = (
        directory
        / 'predictions.partial.json'
    )

    metadata_path = (
        directory
        / 'predictions.provenance.json'
    )


    # ============================================================
    # EXPECTED INFERENCE PROVENANCE
    # ============================================================

    expected = {
        'slot':
            slot,

        'selected_sha256':
            sha256_file(
                selected_path
            ),

        'method_sha256':
            method_sha,

        'training_dataset_fingerprint':
            selected_state[
                'dataset_fingerprint'
            ],

        'official_test_file_sha256':
            sha256_file(
                data[
                    'paths'
                ][
                    'fold3-text.json'
                ]
            ),

        'test_ids_sha256':
            hashlib.sha256(
                stable_json(
                    sorted(
                        data[
                            'test_text'
                        ]
                    )
                ).encode()
            ).hexdigest(),

        'config':
            dict(config),

        'retry_rounds':
            retry_rounds,
    }


    # ============================================================
    # VALIDATE EXISTING PROVENANCE
    # ============================================================

    if metadata_path.exists():

        old_metadata = load_json(
            metadata_path
        )

        if old_metadata != expected:

            raise RuntimeError(
                f'Slot {slot}: '
                f'partial prediction provenance changed'
            )


    # ============================================================
    # LOAD EXISTING PARTIAL PREDICTIONS
    # ============================================================

    if partial_path.exists():

        if not metadata_path.exists():

            raise RuntimeError(
                f'Slot {slot}: '
                f'partial predictions have no provenance record'
            )


        predictions = load_json(
            partial_path
        )


        if (
            not isinstance(
                predictions,
                dict
            )
            or set(predictions)
            - set(
                data[
                    'test_text'
                ]
            )
        ):

            raise ValueError(
                f'Slot {slot}: '
                f'invalid partial prediction IDs'
            )


        # Validate all already completed IDs.
        for identifier, spans in predictions.items():

            validate_prediction_map(
                {
                    identifier:
                        spans
                },
                {
                    identifier:
                        data[
                            'test_text'
                        ][
                            identifier
                        ]
                },
                data[
                    'labels'
                ],
            )


    else:

        predictions = {}

        atomic_json(
            metadata_path,
            expected
        )


    # ============================================================
    # IMPORTANT:
    # PRE-FILL ALL EMPTY DOCUMENTS WITH []
    #
    # This is the key fix.
    # ============================================================

    empty_ids = [
        identifier
        for identifier, text
        in data[
            'test_text'
        ].items()
        if text == ''
    ]


    newly_added_empty = 0


    for identifier in empty_ids:

        if identifier not in predictions:

            predictions[
                identifier
            ] = []

            newly_added_empty += 1


    if newly_added_empty:

        atomic_json(
            partial_path,
            predictions
        )


    if empty_ids:

        print(
            f'Slot {slot}: '
            f'{len(empty_ids)} empty official test '
            f'document(s) detected.'
        )

        print(
            '  Empty documents are assigned [] '
            'without calling the model.'
        )

        print(
            f'  IDs: {empty_ids}'
        )


    # ============================================================
    # INFERENCE RETRY LOOP
    # ============================================================

    failures = {}

    new_successes = 0


    for round_index in range(
        retry_rounds
    ):

        pending = [
            identifier
            for identifier
            in data[
                'test_text'
            ]
            if identifier
            not in predictions
        ]


        if not pending:
            break


        print(
            f'Slot {slot}: '
            f'prediction retry round '
            f'{round_index + 1}, '
            f'{len(pending)} missing IDs'
        )


        for identifier in pending:

            # ----------------------------------------------------
            # SESSION BUDGET
            # ----------------------------------------------------

            if (
                session_deadline is not None
                and time.monotonic()
                >= session_deadline
            ):

                export_recovery(
                    slot
                )

                raise RuntimeError(
                    f'Slot {slot}: '
                    f'inference session budget reached; '
                    f'attach recovery and resume'
                )


            started = time.monotonic()


            try:

                text = data[
                    'test_text'
                ][
                    identifier
                ]


                # ------------------------------------------------
                # SAFETY GUARD:
                # Empty input must never reach tokenizer/model.
                # ------------------------------------------------

                if text == '':

                    spans = []


                else:

                    with torch.inference_mode():

                        spans = method.predict_spans(
                            model,
                            text,
                            tokenizer,
                            data[
                                'labels'
                            ],
                            dict(
                                config
                            ),
                        )


                # ------------------------------------------------
                # VALIDATE ONE DOCUMENT
                # ------------------------------------------------

                validate_prediction_map(
                    {
                        identifier:
                            spans
                    },
                    {
                        identifier:
                            text
                    },
                    data[
                        'labels'
                    ],
                )


                # ------------------------------------------------
                # SAVE SUCCESS
                # ------------------------------------------------

                predictions[
                    identifier
                ] = spans


                atomic_json(
                    partial_path,
                    predictions
                )


                new_successes += 1


                if (
                    new_successes
                    % INFERENCE_RECOVERY_INTERVAL
                    == 0
                ):

                    export_recovery(
                        slot
                    )


                failures.pop(
                    identifier,
                    None
                )


                append_jsonl(
                    directory
                    / 'inference_attempts.jsonl',

                    {
                        'id':
                            identifier,

                        'round':
                            round_index,

                        'status':
                            'success',

                        'seconds':
                            time.monotonic()
                            - started,
                    },
                )


            except Exception as exc:

                failures[
                    identifier
                ] = (
                    f'{type(exc).__name__}: '
                    f'{exc}'
                )


                append_jsonl(
                    directory
                    / 'inference_attempts.jsonl',

                    {
                        'id':
                            identifier,

                        'round':
                            round_index,

                        'status':
                            'failure',

                        'error':
                            failures[
                                identifier
                            ],

                        'seconds':
                            time.monotonic()
                            - started,
                    },
                )


                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


        atomic_json(
            directory
            / 'inference_failures.json',

            failures,
        )


    # ============================================================
    # FINAL COMPLETENESS CHECK
    # ============================================================

    missing = (
        set(
            data[
                'test_text'
            ]
        )
        - set(
            predictions
        )
    )


    if missing:

        export_recovery(
            slot
        )

        raise RuntimeError(
            f'Slot {slot}: '
            f'{len(missing)} test IDs remain '
            f'unpredicted; partial file is resumable.\n'
            f'Missing IDs: {sorted(missing)[:20]}'
        )


    # ============================================================
    # FINAL STRUCTURAL VALIDATION
    # ============================================================

    validate_prediction_map(
        predictions,
        data[
            'test_text'
        ],
        data[
            'labels'
        ],
    )


    print(
        f'Slot {slot}: all '
        f'{len(predictions)} test documents '
        f'predicted successfully.'
    )


    return (
        predictions,
        expected,
    )


print(
    'FINAL Phase-2 empty-document compatibility patch: READY'
)

In [ ]:
# ================================================================
# FINAL GPU-STABILITY PATCH FOR PHASE-2
#
# Tesla T4 = compute capability 7.5
# T4 does NOT have native BF16 support.
#
# The original notebook therefore selected FP16 + GradScaler.
# Its smoke test and training loop incorrectly treated a normal
# FP16 overflow as a fatal nonfinite-gradient error.
#
# Stable policy used here:
#
#   Ampere/newer GPU with BF16 support -> BF16 autocast
#   Tesla T4 / older GPU              -> full FP32
#
# This overrides:
#   choose_gpu_runtime()
#   run_gpu_smoke()
#   train_one_experiment()
#
# Paste this whole cell BEFORE the final
# "Resumable Phase-2 test predictions..." cell.
# ================================================================

from contextlib import nullcontext


# ================================================================
# GPU PRECISION POLICY
# ================================================================

def choose_gpu_runtime() -> dict[str, Any]:

    if not torch.cuda.is_available():

        raise RuntimeError(
            'Phase-2 training requires a Kaggle GPU; '
            'enable the accelerator'
        )


    ids = list(
        range(
            torch.cuda.device_count()
        )
    )


    capabilities = [
        torch.cuda.get_device_capability(i)
        for i in ids
    ]


    names = [
        torch.cuda.get_device_name(i)
        for i in ids
    ]


    # ------------------------------------------------------------
    # BF16 is safe only when ALL visible GPUs support it.
    # Ampere = compute capability >= 8.
    # ------------------------------------------------------------

    bf16_supported = (
        all(
            major >= 8
            for major, _minor
            in capabilities
        )
        and torch.cuda.is_bf16_supported()
    )


    if bf16_supported:

        precision = 'bf16'

        amp_dtype = torch.bfloat16

        use_autocast = True

        scaler_enabled = False


    else:

        # --------------------------------------------------------
        # IMPORTANT:
        # Tesla T4 falls here.
        #
        # Do NOT use FP16 for this training pipeline because the
        # current model/loss can overflow before GradScaler settles.
        #
        # Full FP32 is slower but substantially safer.
        # --------------------------------------------------------

        precision = 'fp32'

        amp_dtype = None

        use_autocast = False

        scaler_enabled = False


    runtime = {

        'ids':
            ids,

        'primary':
            'cuda:0',

        'precision':
            precision,

        'amp_dtype':
            amp_dtype,

        'use_autocast':
            use_autocast,

        'scaler_enabled':
            scaler_enabled,

        'names':
            names,

        'capabilities':
            capabilities,
    }


    print()

    print(
        'GPU runtime selected:'
    )

    for index, name in enumerate(
        names
    ):

        print(
            f'  cuda:{index}: '
            f'{name} '
            f'(capability '
            f'{capabilities[index][0]}.'
            f'{capabilities[index][1]})'
        )


    print(
        f'  Precision: {precision}'
    )


    if precision == 'fp32':

        print(
            '  FP32 selected for stability on '
            'Tesla T4 / pre-Ampere GPU.'
        )


    print()


    return runtime


# ================================================================
# PRECISION CONTEXT
# ================================================================

def phase2_precision_context(
    gpu: Mapping[str, Any]
):

    if gpu.get(
        'use_autocast',
        False
    ):

        return torch.autocast(
            device_type='cuda',
            dtype=gpu[
                'amp_dtype'
            ],
        )


    return nullcontext()


# ================================================================
# FINITE PARAMETER / GRADIENT DIAGNOSTICS
# ================================================================

def _assert_finite_model_parameters(
    model: torch.nn.Module,
    *,
    prefix: str,
):

    bad = []


    for name, parameter in model.named_parameters():

        if not torch.isfinite(
            parameter.detach()
        ).all():

            bad.append(
                name
            )


    if bad:

        raise FloatingPointError(
            f'{prefix}: model already contains '
            f'nonfinite parameters: '
            f'{bad[:10]}'
        )


def _finite_gradient_norm(
    model: torch.nn.Module,
    max_norm: float,
) -> float:

    # ------------------------------------------------------------
    # Check individual gradients BEFORE clipping so that an actual
    # model instability is distinguished from AMP overflow.
    # ------------------------------------------------------------

    bad_gradients = []


    for name, parameter in model.named_parameters():

        gradient = parameter.grad

        if gradient is None:
            continue


        if not torch.isfinite(
            gradient
        ).all():

            bad_gradients.append(
                name
            )


    if bad_gradients:

        raise FloatingPointError(
            'Nonfinite gradients remain even in the '
            'stable precision path.\n'
            f'First affected parameters: '
            f'{bad_gradients[:10]}'
        )


    norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        float(max_norm),
    )


    norm_value = float(
        norm.detach().cpu()
        if torch.is_tensor(norm)
        else norm
    )


    if not math.isfinite(
        norm_value
    ):

        raise FloatingPointError(
            'Gradient norm is nonfinite even in FP32/BF16'
        )


    return norm_value


# ================================================================
# GPU SMOKE TEST
# ================================================================

def run_gpu_smoke(
    spec,
    method,
    data,
    train_ids,
    val_ids,
    config,
    directory,
    method_sha,
    identity,
):

    manifest = (
        directory
        / 'gpu_smoke.json'
    )


    # ------------------------------------------------------------
    # REUSE ONLY A SUCCESSFUL SMOKE RECORD WITH SAME IDENTITY
    # ------------------------------------------------------------

    if manifest.exists():

        report = load_json(
            manifest
        )


        if (
            report.get(
                'identity'
            )
            == identity

            and report.get(
                'status'
            )
            == 'PASS'
        ):

            return report


        # Do NOT let an old failed/stale smoke record poison
        # the corrected run.

        print(
            f'Removing stale GPU smoke record: '
            f'{manifest}'
        )

        manifest.unlink(
            missing_ok=True
        )


    gpu = choose_gpu_runtime()


    # ============================================================
    # BUILD TASK
    # ============================================================

    (
        task,
        tokenizer,
        teacher_tokenizer,
        base_meta,
        _teacher_meta,
    ) = build_task(
        method,
        spec[
            'base_id'
        ],
        spec,
        config,
        data[
            'labels'
        ],
        gpu[
            'primary'
        ],
    )


    _assert_finite_model_parameters(
        task,
        prefix=(
            f'Slot {spec["slot"]} '
            f'GPU smoke'
        ),
    )


    # ============================================================
    # ONE TRAINING BATCH
    # ============================================================

    sampler = ResumableBatchSampler(
        len(
            train_ids
        ),
        int(
            config[
                'batch_size'
            ]
        ),
        int(
            config[
                'seed'
            ]
        ),
    )


    loader = create_loader(
        method,
        tokenizer,
        teacher_tokenizer,
        data[
            'labels'
        ],
        config,
        data[
            'train_text'
        ],
        data[
            'train_gold'
        ],
        train_ids,
        sampler,
    )


    started = time.monotonic()


    batch = move_batch(
        next(
            iter(
                loader
            )
        ),
        gpu[
            'primary'
        ],
    )


    batch_prep_seconds = (
        time.monotonic()
        - started
    )


    # ============================================================
    # OPTIMIZER
    # ============================================================

    optimizer, backend = make_optimizer(
        task,
        config,
    )


    scheduler = make_scheduler(
        method,
        optimizer,
        config,
    )


    # FP32/BF16 path does not require GradScaler.
    scaler = make_scaler(
        False
    )


    for gpu_id in gpu[
        'ids'
    ]:

        torch.cuda.reset_peak_memory_stats(
            gpu_id
        )

        torch.cuda.synchronize(
            gpu_id
        )


    # ============================================================
    # FORWARD + BACKWARD
    # ============================================================

    started = time.monotonic()


    task.train()


    optimizer.zero_grad(
        set_to_none=True
    )


    with phase2_precision_context(
        gpu
    ):

        output = task(
            **batch
        )

        loss = task_loss(
            output
        )


    if not torch.isfinite(
        loss.detach()
    ).all():

        raise FloatingPointError(
            f'Slot {spec["slot"]}: '
            f'GPU smoke produced a '
            f'nonfinite forward loss'
        )


    loss.backward()


    gradient_norm = (
        _finite_gradient_norm(
            task,
            float(
                config[
                    'gradient_clip_norm'
                ]
            ),
        )
    )


    optimizer.step()


    scheduler.step()


    _assert_finite_model_parameters(
        task,
        prefix=(
            f'Slot {spec["slot"]} '
            f'GPU smoke after optimizer step'
        ),
    )


    for gpu_id in gpu[
        'ids'
    ]:

        torch.cuda.synchronize(
            gpu_id
        )


    compute_seconds = (
        time.monotonic()
        - started
    )


    # ============================================================
    # CHECKPOINT ROUND-TRIP
    # ============================================================

    with tempfile.TemporaryDirectory() as temporary_name:

        temporary = (
            Path(
                temporary_name
            )
            / 'gpu_smoke.pt'
        )


        payload = checkpoint_payload(
            task,
            optimizer,
            scheduler,
            None,
            sampler,
            step=1,
            best_score=None,
            identity=identity,
            history_path=(
                directory
                / 'not_yet_created_history.jsonl'
            ),
            config=config,
            base_meta=base_meta,
            method_sha=method_sha,
            dataset_fingerprint=(
                data[
                    'fingerprint'
                ]
            ),
        )


        atomic_torch_save(
            temporary,
            payload
        )


        restore_training_checkpoint(
            temporary,
            task,
            optimizer,
            scheduler,
            None,
            sampler,
            identity,
            config,
            method_sha,
            data[
                'fingerprint'
            ],
        )


        if not optimizer.state:

            raise AssertionError(
                'GPU smoke did not restore '
                'optimizer state'
            )


    # ============================================================
    # HELD-OUT TOKENIZER OFFSET TEST
    # ============================================================

    probe_id = val_ids[
        0
    ]


    probe_text = data[
        'train_text'
    ][
        probe_id
    ]


    encoding = tokenizer.backend.encode(
        probe_text,
        add_special_tokens=False,
    )


    if (
        len(
            encoding.ids
        )
        != len(
            encoding.offsets
        )
    ):

        raise AssertionError(
            'BPE token/offset lengths differ'
        )


    if any(
        not (
            0
            <= start
            <= end
            <= len(
                probe_text
            )
        )

        for start, end
        in encoding.offsets
    ):

        raise AssertionError(
            'BPE offset map is invalid for '
            'a held-out legal document'
        )


    # ============================================================
    # HELD-OUT PREDICTION CONTRACT
    # ============================================================

    task.eval()


    with torch.inference_mode():

        probe_spans = method.predict_spans(
            task,
            probe_text,
            tokenizer,
            data[
                'labels'
            ],
            dict(
                config
            ),
        )


    validate_prediction_map(
        {
            probe_id:
                probe_spans
        },
        {
            probe_id:
                probe_text
        },
        data[
            'labels'
        ],
    )


    # ============================================================
    # REPORT
    # ============================================================

    report = {

        'status':
            'PASS',

        'identity':
            identity,

        'loss':
            float(
                loss.detach().cpu()
            ),

        'gradient_norm':
            gradient_norm,

        'batch_prep_seconds':
            batch_prep_seconds,

        'single_microbatch_compute_seconds':
            compute_seconds,

        'precision':
            gpu[
                'precision'
            ],

        'gpu_names':
            gpu[
                'names'
            ],

        'peak_memory_gib': {

            str(i):
                torch.cuda.max_memory_allocated(
                    i
                )
                / 2**30

            for i
            in gpu[
                'ids'
            ]
        },

        'optimizer_backend':
            backend,

        'checkpoint_restore':
            'PASS',

        'heldout_prediction_contract':
            'PASS',

        'timing_note':
            (
                'One microbatch, not a full '
                'accumulated optimizer step'
            ),
    }


    atomic_json(
        manifest,
        report
    )


    print(
        f'Slot {spec["slot"]} '
        f'GPU smoke PASS '
        f'using {gpu["precision"]}.'
    )


    del (
        task,
        optimizer,
        scheduler,
        scaler,
        batch,
    )


    torch.cuda.empty_cache()


    return report


# ================================================================
# CORRECTED TRAINING LOOP
# ================================================================

def train_one_experiment(
    spec,
    data,
    train_ids,
    val_ids,
    session_deadline,
):

    slot = int(
        spec[
            'slot'
        ]
    )


    directory = experiment_directory(
        slot
    )


    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    attached_recovery = restore_attached_recovery(
        slot
    )


    config = validate_training_config(
        slot
    )


    method, method_sha = load_student_method(
        spec,
        directory
        / spec[
            'module_filename'
        ],
    )


    gpu = choose_gpu_runtime()


    # ============================================================
    # BUILD TASK
    # ============================================================

    (
        task,
        tokenizer,
        teacher_tokenizer,
        base_meta,
        teacher_meta,
    ) = build_task(
        method,
        spec[
            'base_id'
        ],
        spec,
        config,
        data[
            'labels'
        ],
        gpu[
            'primary'
        ],
    )


    _assert_finite_model_parameters(
        task,
        prefix=f'Slot {slot}',
    )


    # ============================================================
    # EXPERIMENT IDENTITY
    # ============================================================

    identity = experiment_identity(
        spec,
        config,
        {
            'student':
                base_meta,

            'teacher':
                teacher_meta,
        },
        data[
            'fingerprint'
        ],
        method_sha,
    )


    identity_path = (
        directory
        / 'experiment_identity.json'
    )


    identity_record = {

        'identity':
            identity,

        'spec':
            dict(
                spec
            ),

        'config':
            config,

        'base_meta':
            base_meta,

        'teacher_meta':
            teacher_meta,

        'dataset_fingerprint':
            data[
                'fingerprint'
            ],

        'method_sha256':
            method_sha,
    }


    if identity_path.exists():

        old_identity = load_json(
            identity_path
        )


        if old_identity != identity_record:

            raise RuntimeError(
                f'Slot {slot}: '
                f'experiment identity changed '
                f'after initialization'
            )


    atomic_json(
        identity_path,
        identity_record,
    )


    if attached_recovery is not None:

        print(
            f'Slot {slot}: '
            f'restored verified recovery '
            f'from {attached_recovery}'
        )


    latest = (
        directory
        / 'checkpoint.latest.pt'
    )


    history = (
        directory
        / 'training.jsonl'
    )


    # ============================================================
    # DATA LOADER
    # ============================================================

    sampler = ResumableBatchSampler(
        len(
            train_ids
        ),
        int(
            config[
                'batch_size'
            ]
        ),
        int(
            config[
                'seed'
            ]
        ),
    )


    loader = create_loader(
        method,
        tokenizer,
        teacher_tokenizer,
        data[
            'labels'
        ],
        config,
        data[
            'train_text'
        ],
        data[
            'train_gold'
        ],
        train_ids,
        sampler,
    )


    # ============================================================
    # OPTIMIZER
    # ============================================================

    optimizer, backend = make_optimizer(
        task,
        config,
    )


    scheduler = make_scheduler(
        method,
        optimizer,
        config,
    )


    # FP32/BF16 stable path.
    scaler = make_scaler(
        False
    )


    best_score = -math.inf

    step = 0

    state = None


    # ============================================================
    # RESUME
    # ============================================================

    if latest.exists():

        if RUN_MODE == 'fresh':

            raise RuntimeError(
                f'Slot {slot}: '
                f'fresh mode found a compatible '
                f'checkpoint; choose resume'
            )


        state = restore_training_checkpoint(
            latest,
            task,
            optimizer,
            scheduler,
            None,
            sampler,
            identity,
            config,
            method_sha,
            data[
                'fingerprint'
            ],
        )


        step = int(
            state[
                'step'
            ]
        )


        saved_best = state[
            'best_validation_score'
        ]


        best_score = (
            float(
                saved_best
            )
            if saved_best
            is not None
            else -math.inf
        )


        reconcile_history(
            history,
            step
        )


        print(
            f'Slot {slot}: '
            f'resuming optimizer step {step}'
        )


    # ============================================================
    # NEW TRAINING RUN -> GPU SMOKE
    # ============================================================

    else:

        if (
            (
                RUN_MODE == 'resume'
                and slot
                in (
                    RESUME_EXPECTED_SLOTS
                    or set()
                )
            )
            or history.exists()
            or (
                directory
                / 'selected_model.pt'
            ).exists()
        ):

            raise RuntimeError(
                f'Slot {slot}: '
                f'no latest checkpoint; refusing '
                f'a silent step-zero restart'
            )


        del (
            task,
            optimizer,
            scheduler,
            scaler,
            loader,
        )


        torch.cuda.empty_cache()


        smoke = run_gpu_smoke(
            spec,
            method,
            data,
            train_ids,
            val_ids,
            config,
            directory,
            method_sha,
            identity,
        )


        print(
            f'Slot {slot} GPU smoke:',
            smoke,
        )


        (
            task,
            tokenizer,
            teacher_tokenizer,
            base_meta,
            teacher_meta,
        ) = build_task(
            method,
            spec[
                'base_id'
            ],
            spec,
            config,
            data[
                'labels'
            ],
            gpu[
                'primary'
            ],
        )


        sampler = ResumableBatchSampler(
            len(
                train_ids
            ),
            int(
                config[
                    'batch_size'
                ]
            ),
            int(
                config[
                    'seed'
                ]
            ),
        )


        loader = create_loader(
            method,
            tokenizer,
            teacher_tokenizer,
            data[
                'labels'
            ],
            config,
            data[
                'train_text'
            ],
            data[
                'train_gold'
            ],
            train_ids,
            sampler,
        )


        optimizer, backend = make_optimizer(
            task,
            config,
        )


        scheduler = make_scheduler(
            method,
            optimizer,
            config,
        )


        scaler = make_scaler(
            False
        )


    iterator = iter(
        loader
    )


    if (
        latest.exists()
        and step
        and state is not None
    ):

        restore_rng(
            state[
                'rng_state'
            ]
        )


    # ============================================================
    # ALREADY COMPLETE
    # ============================================================

    if step >= int(
        config[
            'max_steps'
        ]
    ):

        if not (
            directory
            / 'selected_model.pt'
        ).exists():

            raise RuntimeError(
                f'Slot {slot}: '
                f'training marked complete but '
                f'selected checkpoint is absent'
            )


        completed = {

            'slot':
                slot,

            'status':
                'complete',

            'step':
                step,

            'target_step':
                int(
                    config[
                        'max_steps'
                    ]
                ),

            'selected':
                str(
                    directory
                    / 'selected_model.pt'
                ),
        }


        atomic_json(
            directory
            / 'training_result.json',
            completed,
        )


        export_recovery(
            slot
        )


        return completed


    # ============================================================
    # SESSION DEADLINE
    # ============================================================

    if time.monotonic() >= session_deadline:

        if latest.exists():

            export_recovery(
                slot
            )


        return {

            'slot':
                slot,

            'status':
                'time_budget',

            'step':
                step,
        }


    started = time.monotonic()


    # ============================================================
    # TRAIN
    # ============================================================

    try:

        while step < int(
            config[
                'max_steps'
            ]
        ):

            if time.monotonic() >= session_deadline:
                break


            step_started = time.monotonic()


            task.train()


            optimizer.zero_grad(
                set_to_none=True
            )


            loss_total = 0.0


            for _ in range(
                int(
                    config[
                        'gradient_accumulation_steps'
                    ]
                )
            ):

                try:

                    batch = next(
                        iterator
                    )


                except StopIteration:

                    sampler.next_epoch()

                    iterator = iter(
                        loader
                    )

                    batch = next(
                        iterator
                    )


                batch = move_batch(
                    batch,
                    gpu[
                        'primary'
                    ],
                )


                with phase2_precision_context(
                    gpu
                ):

                    loss = task_loss(
                        task(
                            **batch
                        )
                    )


                    backward_loss = (
                        loss
                        / int(
                            config[
                                'gradient_accumulation_steps'
                            ]
                        )
                    )


                if not torch.isfinite(
                    loss.detach()
                ).all():

                    raise FloatingPointError(
                        f'Slot {slot}: '
                        f'nonfinite forward loss'
                    )


                backward_loss.backward()


                loss_total += float(
                    loss.detach().cpu()
                )


            # ====================================================
            # GRADIENT VALIDATION + CLIPPING
            # ====================================================

            norm = _finite_gradient_norm(
                task,
                float(
                    config[
                        'gradient_clip_norm'
                    ]
                ),
            )


            # ====================================================
            # OPTIMIZER STEP
            # ====================================================

            optimizer.step()


            scheduler.step()


            _assert_finite_model_parameters(
                task,
                prefix=(
                    f'Slot {slot} '
                    f'after optimizer step '
                    f'{step + 1}'
                ),
            )


            step += 1


            validation = None

            improved = False


            # ====================================================
            # VALIDATION
            # ====================================================

            if (
                step
                % int(
                    config[
                        'validation_interval'
                    ]
                )
                == 0

                or step
                == int(
                    config[
                        'max_steps'
                    ]
                )
            ):

                cap = int(
                    config.get(
                        'validation_document_cap',
                        len(
                            val_ids
                        ),
                    )
                )


                validation_ids = (
                    val_ids
                    if step
                    == int(
                        config[
                            'max_steps'
                        ]
                    )
                    else val_ids[
                        :cap
                    ]
                )


                validation = evaluate_student_method(
                    method,
                    task,
                    tokenizer,
                    data,
                    validation_ids,
                    config,
                    gpu[
                        'primary'
                    ],
                )


                validation[
                    'evaluated_documents'
                ] = len(
                    validation_ids
                )


                validation[
                    'full_validation'
                ] = (
                    len(
                        validation_ids
                    )
                    == len(
                        val_ids
                    )
                )


                atomic_json(
                    directory
                    / (
                        f'validation_step_'
                        f'{step:07d}.json'
                    ),
                    validation,
                )


                score = float(
                    validation[
                        'strict_macro_f1_local'
                    ]
                )


                improved = (
                    score
                    > best_score
                )


                best_score = max(
                    best_score,
                    score,
                )


            # ====================================================
            # TIMING
            # ====================================================

            for gpu_id in gpu[
                'ids'
            ]:

                torch.cuda.synchronize(
                    gpu_id
                )


            optimizer_step_seconds = (
                time.monotonic()
                - step_started
            )


            row = {

                'step':
                    step,

                'epoch':
                    sampler.epoch,

                'loss':
                    (
                        loss_total
                        / int(
                            config[
                                'gradient_accumulation_steps'
                            ]
                        )
                    ),

                'validation_macro_f1':
                    (
                        None
                        if validation is None
                        else validation[
                            'strict_macro_f1_local'
                        ]
                    ),

                'learning_rate':
                    optimizer.param_groups[
                        0
                    ][
                        'lr'
                    ],

                'gradient_norm':
                    norm,

                'seconds_elapsed':
                    (
                        time.monotonic()
                        - started
                    ),

                'optimizer_step_seconds':
                    optimizer_step_seconds,

                'nominal_documents_per_second':
                    (
                        int(
                            config[
                                'batch_size'
                            ]
                        )
                        * int(
                            config[
                                'gradient_accumulation_steps'
                            ]
                        )
                        / max(
                            optimizer_step_seconds,
                            1e-9,
                        )
                    ),

                'gpu_memory_gib': {

                    str(i):
                        (
                            torch.cuda
                            .max_memory_allocated(
                                i
                            )
                            / 2**30
                        )

                    for i
                    in gpu[
                        'ids'
                    ]
                },

                'optimizer_backend':
                    backend,

                'precision':
                    gpu[
                        'precision'
                    ],
            }


            append_jsonl(
                history,
                row,
            )


            # ====================================================
            # CHECKPOINT
            # ====================================================

            if (
                step
                % int(
                    config[
                        'checkpoint_interval'
                    ]
                )
                == 0

                or improved

                or step
                == int(
                    config[
                        'max_steps'
                    ]
                )
            ):

                payload = checkpoint_payload(
                    task,
                    optimizer,
                    scheduler,
                    None,
                    sampler,
                    step=step,
                    best_score=best_score,
                    identity=identity,
                    history_path=history,
                    config=config,
                    base_meta=base_meta,
                    method_sha=method_sha,
                    dataset_fingerprint=(
                        data[
                            'fingerprint'
                        ]
                    ),
                )


                atomic_torch_save(
                    latest,
                    payload,
                )


                if improved:

                    save_selected_model(
                        directory,
                        task,
                        step=step,
                        score=best_score,
                        identity=identity,
                        config=config,
                        method_sha=method_sha,
                        base_meta=base_meta,
                        tokenizer_fingerprint=(
                            base_meta[
                                'tokenizer_fingerprint'
                            ]
                        ),
                        dataset_fingerprint=(
                            data[
                                'fingerprint'
                            ]
                        ),
                    )


                if (
                    step
                    % int(
                        config[
                            'recovery_interval'
                        ]
                    )
                    == 0
                ):

                    export_recovery(
                        slot
                    )


        # ========================================================
        # FINAL CHECKPOINT
        # ========================================================

        needs_final_checkpoint = True


        if latest.exists():

            existing = torch.load(
                latest,
                map_location='cpu',
                weights_only=True,
            )


            if int(
                existing[
                    'step'
                ]
            ) == step:

                needs_final_checkpoint = False


        if needs_final_checkpoint:

            payload = checkpoint_payload(
                task,
                optimizer,
                scheduler,
                None,
                sampler,
                step=step,
                best_score=best_score,
                identity=identity,
                history_path=history,
                config=config,
                base_meta=base_meta,
                method_sha=method_sha,
                dataset_fingerprint=(
                    data[
                        'fingerprint'
                    ]
                ),
            )


            atomic_torch_save(
                latest,
                payload,
            )


        # ========================================================
        # RESULT
        # ========================================================

        result = {

            'slot':
                slot,

            'status':
                (
                    'complete'
                    if step
                    == int(
                        config[
                            'max_steps'
                        ]
                    )
                    else 'time_budget'
                ),

            'step':
                step,

            'target_step':
                int(
                    config[
                        'max_steps'
                    ]
                ),

            'best_strict_macro_f1_local':
                (
                    None
                    if best_score
                    == -math.inf
                    else best_score
                ),

            'checkpoint':
                str(
                    latest
                ),

            'recovery':
                str(
                    directory
                    / recovery_name(
                        slot
                    )
                ),

            'selected':
                str(
                    directory
                    / 'selected_model.pt'
                ),
        }


        atomic_json(
            directory
            / 'training_result.json',
            result,
        )


        export_recovery(
            slot
        )


        return result


    except Exception as exc:

        append_jsonl(
            directory
            / 'failure_events.jsonl',

            {
                'last_committed_step':
                    step,

                'error':
                    (
                        f'{type(exc).__name__}: '
                        f'{exc}'
                    ),
            },
        )


        if latest.exists():

            export_recovery(
                slot
            )


        raise


print()
print(
    'FINAL GPU-STABILITY PATCH: READY'
)

print(
    'Tesla T4 will use FP32 instead of unstable FP16.'
)

print()

In [ ]:
def load_selected_task(spec, data, config, method, method_sha, device):
    slot = int(spec['slot'])
    directory = experiment_directory(slot)
    selected_path = directory / 'selected_model.pt'
    if not selected_path.is_file():
        raise FileNotFoundError(f'Slot {slot}: selected trained checkpoint missing')
    selected = torch.load(selected_path, map_location='cpu', weights_only=True)
    if selected.get('format') != 'cs781_phase2_selected_v1' or selected.get('method_sha256') != method_sha:
        raise RuntimeError(f'Slot {slot}: selected checkpoint provenance mismatch')
    if ((data['fingerprint'] is not None and selected.get('dataset_fingerprint') != data['fingerprint'])
            or selected.get('training_config') != dict(config)):
        raise RuntimeError(f'Slot {slot}: selected checkpoint data/config mismatch')
    task, tokenizer, _teacher_tokenizer, base_meta, _teacher_meta = build_task(
        method, spec['base_id'], spec, config, data['labels'], device
    )
    if selected.get('base_meta') != base_meta or selected.get('tokenizer_fingerprint') != base_meta['tokenizer_fingerprint']:
        raise RuntimeError(f'Slot {slot}: selected checkpoint Phase-1 base mismatch')
    bare_model(task).load_state_dict(selected['model_state'], strict=True)
    task.eval()
    return task, tokenizer, selected_path, selected


def generate_test_predictions(spec, data, config, method, method_sha, *, retry_rounds=3,
                              session_deadline=None):
    slot = int(spec['slot'])
    directory = experiment_directory(slot)
    model, tokenizer, selected_path, selected_state = load_selected_task(
        spec, data, config, method, method_sha, 'cuda:0'
    )
    partial_path = directory / 'predictions.partial.json'
    metadata_path = directory / 'predictions.provenance.json'
    expected = {'slot': slot, 'selected_sha256': sha256_file(selected_path),
                'method_sha256': method_sha,
                'training_dataset_fingerprint': selected_state['dataset_fingerprint'],
                'official_test_file_sha256': sha256_file(data['paths']['fold3-text.json']),
                'test_ids_sha256': hashlib.sha256(stable_json(sorted(data['test_text'])).encode()).hexdigest(),
                'config': dict(config), 'retry_rounds': retry_rounds}
    if metadata_path.exists() and load_json(metadata_path) != expected:
        raise RuntimeError(f'Slot {slot}: partial prediction provenance changed')
    if partial_path.exists():
        if not metadata_path.exists():
            raise RuntimeError(f'Slot {slot}: partial predictions have no provenance record')
        predictions = load_json(partial_path)
        if not isinstance(predictions, dict) or set(predictions) - set(data['test_text']):
            raise ValueError(f'Slot {slot}: invalid partial prediction IDs')
        for identifier, spans in predictions.items():
            validate_prediction_map({identifier: spans},
                                    {identifier: data['test_text'][identifier]}, data['labels'])
    else:
        predictions = {}
        atomic_json(metadata_path, expected)
    failures = {}
    new_successes = 0
    for round_index in range(retry_rounds):
        pending = [identifier for identifier in data['test_text'] if identifier not in predictions]
        if not pending:
            break
        print(f'Slot {slot}: prediction retry round {round_index + 1}, {len(pending)} missing IDs')
        for identifier in pending:
            if session_deadline is not None and time.monotonic() >= session_deadline:
                export_recovery(slot)
                raise RuntimeError(f'Slot {slot}: inference session budget reached; attach recovery and resume')
            started = time.monotonic()
            try:
                with torch.inference_mode():
                    spans = method.predict_spans(model, data['test_text'][identifier],
                                                 tokenizer, data['labels'], dict(config))
                validate_prediction_map({identifier: spans},
                                        {identifier: data['test_text'][identifier]}, data['labels'])
                predictions[identifier] = spans
                atomic_json(partial_path, predictions)
                new_successes += 1
                if new_successes % INFERENCE_RECOVERY_INTERVAL == 0:
                    export_recovery(slot)
                failures.pop(identifier, None)
                append_jsonl(directory / 'inference_attempts.jsonl',
                             {'id': identifier, 'round': round_index, 'status': 'success',
                              'seconds': time.monotonic() - started})
            except Exception as exc:
                failures[identifier] = f'{type(exc).__name__}: {exc}'
                append_jsonl(directory / 'inference_attempts.jsonl',
                             {'id': identifier, 'round': round_index, 'status': 'failure',
                              'error': failures[identifier], 'seconds': time.monotonic() - started})
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        atomic_json(directory / 'inference_failures.json', failures)
    missing = set(data['test_text']) - set(predictions)
    if missing:
        export_recovery(slot)
        raise RuntimeError(f'Slot {slot}: {len(missing)} test IDs remain unpredicted; partial file is resumable')
    validate_prediction_map(predictions, data['test_text'], data['labels'])
    return predictions, expected


def make_submission_zip(slot: int, predictions: Mapping[str, Any], data, provenance) -> dict[str, Any]:
    directory = experiment_directory(slot)
    report = validate_prediction_map(predictions, data['test_text'], data['labels'])
    output = directory / 'submission.zip'
    prediction_path = directory / 'prediction.json'
    atomic_json(prediction_path, predictions)
    with tempfile.NamedTemporaryFile(dir=directory, suffix='.zip', delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, 'prediction.json')
        with zipfile.ZipFile(temporary) as archive:
            if archive.namelist() != ['prediction.json'] or archive.testzip() is not None:
                raise ValueError('Phase-2 ZIP structure/CRC failed')
            inside = json.loads(archive.read('prediction.json'))
            validate_prediction_map(inside, data['test_text'], data['labels'])
        os.replace(temporary, output)
    finally:
        temporary.unlink(missing_ok=True)
    result = {**report, 'slot': slot, 'zip_path': str(output), 'zip_sha256': sha256_file(output),
              'prediction_sha256': sha256_file(prediction_path), 'provenance': provenance}
    atomic_json(directory / 'submission_validation.json', result)
    return result


def prepare_phase2_data():
    data = load_official_phase2()
    if VALIDATION_FRACTION is None:
        raise RuntimeError('Student must choose VALIDATION_FRACTION before training')
    train_ids, val_ids = deterministic_split(list(data['train_text']), float(VALIDATION_FRACTION), SEED)
    manifest = {'dataset_fingerprint': data['fingerprint'], 'train_ids': train_ids,
                'validation_ids': val_ids, 'seed': SEED,
                'student_validation_fraction': VALIDATION_FRACTION,
                'official_test_excluded_from_training': True}
    path = WORK_ROOT / 'phase2_split_manifest.json'
    if path.exists() and load_json(path) != manifest:
        raise RuntimeError('Existing Phase-2 validation split differs; use a new workspace')
    atomic_json(path, manifest)
    return data, train_ids, val_ids




In [ ]:
IMPROVED_COMMON_SOURCE = '"""Shared, self-contained Phase-2 span tagger used by the six method modules.\n\nAI-assisted reference implementation: disclose its use under the course rules.\nNo official test labels or outside pretrained weights are used here.\n"""\nfrom __future__ import annotations\n\nimport math\nimport random\nfrom functools import lru_cache\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef _encode(tokenizer, text):\n    result = tokenizer.backend.encode(text, add_special_tokens=False)\n    return tuple(result.ids), tuple((int(a), int(b)) for a, b in result.offsets)\n\n\n@lru_cache(maxsize=4096)\ndef _cached_encode(tokenizer_key, text, tokenizer):\n    return _encode(tokenizer, text)\n\n\ndef encode(tokenizer, text):\n    # The backend identity keeps independently trained Phase-1 BPEs separate.\n    return _cached_encode(id(tokenizer.backend), text, tokenizer)\n\n\ndef _tag_id(label_index, kind):\n    return 1 + 4 * label_index + {\'B\': 0, \'I\': 1, \'L\': 2, \'U\': 3}[kind]\n\n\ndef _to_tags(offsets, spans, labels):\n    tags = [0] * len(offsets)\n    for start, end, name in spans:\n        if name not in labels:\n            continue\n        indices = [i for i, (a, b) in enumerate(offsets) if b > start and a < end and a < b]\n        if not indices:\n            continue\n        label_index = labels.index(name)\n        if len(indices) == 1:\n            tags[indices[0]] = _tag_id(label_index, \'U\')\n        else:\n            tags[indices[0]] = _tag_id(label_index, \'B\')\n            for i in indices[1:-1]:\n                tags[i] = _tag_id(label_index, \'I\')\n            tags[indices[-1]] = _tag_id(label_index, \'L\')\n    return tags\n\n\ndef _boundary_targets(offsets, spans, max_shift):\n    starts, ends = [-100] * len(offsets), [-100] * len(offsets)\n    for start, end, _name in spans:\n        indices = [i for i, (a, b) in enumerate(offsets) if b > start and a < end and a < b]\n        if not indices:\n            continue\n        first, last = indices[0], indices[-1]\n        start_delta = start - offsets[first][0]\n        end_delta = offsets[last][1] - end\n        if 0 <= start_delta <= max_shift:\n            starts[first] = start_delta\n        if 0 <= end_delta <= max_shift:\n            ends[last] = end_delta\n    return starts, ends\n\n\ndef _windows(length, size, stride):\n    if length <= size:\n        return [(0, length)]\n    starts = list(range(0, length - size + 1, stride))\n    if starts[-1] != length - size:\n        starts.append(length - size)\n    return [(start, min(start + size, length)) for start in starts]\n\n\ndef _pad(sequences, pad_value, dtype=torch.long):\n    width = max(len(seq) for seq in sequences)\n    out = torch.full((len(sequences), width), pad_value, dtype=dtype)\n    for row, values in enumerate(sequences):\n        out[row, :len(values)] = torch.tensor(values, dtype=dtype)\n    return out\n\n\ndef collate(rows, tokenizer, labels, config, teacher_tokenizer=None):\n    size, stride = int(config[\'max_tokens\']), int(config[\'window_stride\'])\n    ids_rows, masks, tag_rows = [], [], []\n    start_rows, end_rows = [], []\n    teacher_ids_rows, teacher_maps = [], []\n    for row in rows:\n        text = row[\'text\']\n        ids, offsets = encode(tokenizer, text)\n        if not ids:\n            ids, offsets = (tokenizer.unk_token_id,), ((0, 0),)\n        windows = _windows(len(ids), size, stride)\n        spans = row.get(\'spans\', [])\n        entity_windows = [item for item in windows if any(\n            offsets[item[0]][0] <= a and offsets[item[1] - 1][1] >= b\n            for a, b, _label in spans)]\n        chosen = random.choice(entity_windows if entity_windows and random.random() < .7 else windows)\n        lo, hi = chosen\n        segment_offsets = offsets[lo:hi]\n        # Ignore gold entities cut by a window; overlap makes them available in other windows.\n        contained = [(a, b, name) for a, b, name in spans if\n                     segment_offsets[0][0] <= a and b <= segment_offsets[-1][1]]\n        ids_rows.append(ids[lo:hi])\n        masks.append([1] * (hi - lo))\n        tag_rows.append(_to_tags(segment_offsets, contained, labels))\n        start_targets, end_targets = _boundary_targets(\n            segment_offsets, contained, int(config.get(\'max_char_shift\', 16)))\n        start_rows.append(start_targets)\n        end_rows.append(end_targets)\n        if teacher_tokenizer is not None:\n            char_lo, char_hi = segment_offsets[0][0], segment_offsets[-1][1]\n            teacher_ids, teacher_offsets = encode(teacher_tokenizer, text[char_lo:char_hi])\n            if not teacher_ids:\n                teacher_ids, teacher_offsets = (teacher_tokenizer.unk_token_id,), ((0, 0),)\n            teacher_offsets = [(a + char_lo, b + char_lo) for a, b in teacher_offsets]\n            teacher_ids_rows.append(teacher_ids[:int(config[\'teacher_max_tokens\'])])\n            teacher_offsets = teacher_offsets[:int(config[\'teacher_max_tokens\'])]\n            teacher_maps.append([min(range(len(teacher_offsets)), key=lambda j:\n                                 abs(teacher_offsets[j][0] - a)) for a, _b in segment_offsets])\n    result = {\'input_ids\': _pad(ids_rows, tokenizer.pad_token_id),\n              \'attention_mask\': _pad(masks, 0, torch.bool),\n              \'labels\': _pad(tag_rows, -100),\n              \'start_boundaries\': _pad(start_rows, -100),\n              \'end_boundaries\': _pad(end_rows, -100)}\n    if teacher_tokenizer is not None:\n        result.update(teacher_input_ids=_pad(teacher_ids_rows, teacher_tokenizer.pad_token_id),\n                      teacher_attention_mask=_pad([[1] * len(x) for x in teacher_ids_rows], 0, torch.bool),\n                      teacher_index=_pad(teacher_maps, 0))\n    return result\n\n\nclass AdditiveLoRA(nn.Module):\n    def __init__(self, base, rank, alpha):\n        super().__init__()\n        self.base = base\n        self.a = nn.Linear(base.in_features, rank, bias=False)\n        self.b = nn.Linear(rank, base.out_features, bias=False)\n        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))\n        nn.init.zeros_(self.b.weight)\n        self.scale = alpha / rank\n\n    def forward(self, x):\n        return self.base(x) + self.b(self.a(x)) * self.scale\n\n\nclass MultiplicativeLoRMA(nn.Module):\n    """LoRMA+: y=(I+alpha/r B A) W0 x + frozen original bias."""\n    def __init__(self, base, rank, alpha):\n        super().__init__()\n        self.base = base\n        self.a = nn.Linear(base.out_features, rank, bias=False)\n        self.b = nn.Linear(rank, base.out_features, bias=False)\n        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))\n        nn.init.zeros_(self.b.weight)\n        self.scale = alpha / rank\n\n    def forward(self, x):\n        y = self.base(x)\n        return y + self.b(self.a(F.linear(x, self.base.weight))) * self.scale\n\n\nclass ResidualAdapter(nn.Module):\n    def __init__(self, block, width, bottleneck):\n        super().__init__()\n        self.block = block\n        self.down = nn.Linear(width, bottleneck)\n        self.up = nn.Linear(bottleneck, width)\n        nn.init.zeros_(self.up.weight)\n        nn.init.zeros_(self.up.bias)\n\n    def forward(self, *args, **kwargs):\n        out = self.block(*args, **kwargs)\n        return out + self.up(F.gelu(self.down(out)))\n\n\ndef adapt_base(base, base_id, method, config):\n    if method == \'full_finetune\':\n        return\n    for param in base.parameters():\n        param.requires_grad_(False)\n    if method == \'partial_finetune\':\n        if base_id != \'02\':\n            raise ValueError(\'Partial encoder method requires Phase-1 model 02\')\n        for block in base.encoder_blocks[-int(config[\'unfrozen_layers\']):]:\n            for param in block.parameters():\n                param.requires_grad_(True)\n        for param in base.encoder_final_norm.parameters():\n            param.requires_grad_(True)\n    elif method in {\'lora\', \'lorma\'}:\n        rank, alpha = int(config[\'rank\']), float(config[\'adapter_alpha\'])\n        projection_names = {\'query\', \'value\'} if method == \'lora\' else {\'q_proj\', \'v_proj\'}\n        replaced = 0\n        for block in base.blocks:\n            attention = block.attention\n            for name in projection_names:\n                original = getattr(attention, name, None)\n                if isinstance(original, nn.Linear):\n                    replacement = AdditiveLoRA(original, rank, alpha) if method == \'lora\' else MultiplicativeLoRMA(original, rank, alpha)\n                    setattr(attention, name, replacement)\n                    replaced += 1\n        if not replaced:\n            raise RuntimeError(\'No attention projections were adapted\')\n    elif method == \'adapter\':\n        for i, block in enumerate(base.blocks):\n            base.blocks[i] = ResidualAdapter(block, base.config.hidden_size, int(config[\'adapter_bottleneck\']))\n    elif method == \'knowledge_distillation\':\n        # The smaller Phase-1 student is frozen; its task head learns from gold and teacher.\n        pass\n    else:\n        raise ValueError(f\'Unknown adaptation method: {method}\')\n\n\nclass EntityTagger(nn.Module):\n    def __init__(self, base, base_id, labels, method, config, teacher=None):\n        super().__init__()\n        self.base, self.base_id = base, base_id\n        self.labels = tuple(labels)\n        self.method = method\n        self.config = dict(config)\n        adapt_base(base, base_id, method, config)\n        width = base.hidden_size if hasattr(base, \'hidden_size\') else (\n            base.config.hidden_size if hasattr(base.config, \'hidden_size\') else int(base.config[\'hidden_size\']))\n        self.head = nn.Linear(width, 1 + 4 * len(labels))\n        self.start_boundary_head = nn.Linear(width, int(config.get(\'max_char_shift\', 16)) + 1)\n        self.end_boundary_head = nn.Linear(width, int(config.get(\'max_char_shift\', 16)) + 1)\n        self.dropout = nn.Dropout(float(config.get(\'head_dropout\', .1)))\n        self.teacher = teacher\n        if teacher is not None:\n            teacher.eval()\n            for param in teacher.parameters():\n                param.requires_grad_(False)\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.teacher is not None:\n            self.teacher.eval()\n        return self\n\n    def features(self, input_ids, attention_mask):\n        base = self.base\n        length = input_ids.shape[1]\n        if self.base_id == \'01\':\n            positions = torch.arange(length, device=input_ids.device)\n            x = base.embedding_dropout(base.token_embedding(input_ids) + base.position_embedding(positions)[None])\n            for block in base.blocks:\n                x = block(x)\n            return base.final_norm(x)\n        if self.base_id == \'02\':\n            return base.encode(input_ids, attention_mask)[0]\n        if self.base_id == \'03\':\n            visibility = base._prefix_hybrid_visibility(attention_mask, attention_mask.sum(1))\n            x = base.embedding_dropout(base.token_embedding(input_ids) * math.sqrt(base.hidden_size))\n            cosine, sine = base.rotary(length, x.dtype)\n            for block in base.blocks:\n                x = block(x, visibility, cosine, sine)\n            return base.final_norm(x)\n        if self.base_id == \'04\':\n            x = base.embedding(input_ids)\n            for block in base.blocks:\n                x = block(x, attention_mask)\n            return base.final_norm(x)\n        raise ValueError(self.base_id)\n\n    def forward(self, input_ids, attention_mask, labels=None,\n                start_boundaries=None, end_boundaries=None,\n                teacher_input_ids=None, teacher_attention_mask=None, teacher_index=None):\n        hidden = self.dropout(self.features(input_ids, attention_mask))\n        logits = self.head(hidden)\n        start_logits = self.start_boundary_head(hidden)\n        end_logits = self.end_boundary_head(hidden)\n        if labels is None:\n            return logits, start_logits, end_logits\n        weights = logits.new_ones(logits.shape[-1])\n        weights[0] = float(self.config.get(\'outside_class_weight\', .3))\n        supervised = F.cross_entropy(logits.flatten(0, 1).float(), labels.flatten(),\n                                     weight=weights.float(), ignore_index=-100)\n        if start_boundaries is not None and (start_boundaries != -100).any():\n            supervised = supervised + float(self.config.get(\'boundary_loss_weight\', .2)) * F.cross_entropy(\n                start_logits.flatten(0, 1).float(), start_boundaries.flatten(), ignore_index=-100)\n        if end_boundaries is not None and (end_boundaries != -100).any():\n            supervised = supervised + float(self.config.get(\'boundary_loss_weight\', .2)) * F.cross_entropy(\n                end_logits.flatten(0, 1).float(), end_boundaries.flatten(), ignore_index=-100)\n        if self.teacher is None:\n            return {\'loss\': supervised}\n        if teacher_input_ids is None or teacher_index is None:\n            raise ValueError(\'Distillation batch lacks teacher inputs/alignment\')\n        with torch.no_grad():\n            teacher_logits = self.teacher(teacher_input_ids, teacher_attention_mask)[0].float()\n        teacher_logits = teacher_logits.gather(1, teacher_index.unsqueeze(-1).expand(-1, -1, logits.shape[-1]))\n        temperature = float(self.config[\'distill_temperature\'])\n        per_token = F.kl_div(F.log_softmax(logits.float() / temperature, -1),\n                             F.softmax(teacher_logits / temperature, -1), reduction=\'none\').sum(-1)\n        valid = labels.ne(-100)\n        distill = (per_token * valid).sum() / valid.sum().clamp_min(1) * temperature ** 2\n        mix = float(self.config[\'distill_weight\'])\n        return {\'loss\': (1 - mix) * supervised + mix * distill}\n\n\ndef predict(model, text, tokenizer, labels, config):\n    ids, offsets = encode(tokenizer, text)\n    if not ids:\n        return []\n    size, stride = int(config[\'max_tokens\']), int(config[\'window_stride\'])\n    windows = _windows(len(ids), size, stride)\n    device = next(model.parameters()).device\n    batch_size = int(config.get(\'inference_window_batch_size\', 4))\n    candidates = {}\n    was_training = model.training\n    model.eval()\n    with torch.inference_mode():\n        cursor = 0\n        while cursor < len(windows):\n            group = windows[cursor:cursor + batch_size]\n            batch = [ids[a:b] for a, b in group]\n            input_ids = _pad(batch, tokenizer.pad_token_id).to(device)\n            masks = _pad([[1] * len(row) for row in batch], 0, torch.bool).to(device)\n            try:\n                logits, start_logits, end_logits = model(input_ids, masks)\n            except RuntimeError as exc:\n                if \'out of memory\' not in str(exc).lower() or batch_size == 1 or device.type != \'cuda\':\n                    raise\n                del input_ids, masks\n                torch.cuda.empty_cache()\n                batch_size = max(1, batch_size // 2)\n                continue\n            probabilities = logits.float().softmax(-1)\n            kinds = probabilities.argmax(-1).cpu().tolist()\n            scores = probabilities.max(-1).values.cpu().tolist()\n            start_deltas = start_logits.argmax(-1).cpu().tolist()\n            end_deltas = end_logits.argmax(-1).cpu().tolist()\n            for row, (start_token, end_token) in enumerate(group):\n                predictions = kinds[row][:end_token - start_token]\n                confidence = scores[row][:end_token - start_token]\n                active_start, active_label, active_scores = None, None, []\n\n                def flush(last):\n                    if active_start is None:\n                        return\n                    raw_a = offsets[start_token + active_start][0]\n                    raw_b = offsets[start_token + last][1]\n                    a = raw_a + start_deltas[row][active_start]\n                    b = raw_b - end_deltas[row][last]\n                    if a >= b:\n                        a, b = raw_a, raw_b\n                    a = max(0, min(a, len(text)))\n                    b = max(0, min(b, len(text)))\n                    while a < b and text[a].isspace():\n                        a += 1\n                    while b > a and text[b - 1].isspace():\n                        b -= 1\n                    if a < b:\n                        key = (a, b, labels[active_label])\n                        candidates[key] = max(candidates.get(key, 0), sum(active_scores) / len(active_scores))\n\n                for i, tag in enumerate(predictions):\n                    if tag == 0:\n                        if active_start is not None:\n                            flush(i - 1)\n                        active_start, active_label, active_scores = None, None, []\n                        continue\n                    label, kind = divmod(tag - 1, 4)\n                    if kind == 3:  # U\n                        if active_start is not None:\n                            flush(i - 1)\n                        active_start, active_label, active_scores = i, label, [confidence[i]]\n                        flush(i)\n                        active_start, active_label, active_scores = None, None, []\n                    elif kind == 0 or active_start is None or label != active_label:\n                        if active_start is not None:\n                            flush(i - 1)\n                        active_start, active_label, active_scores = i, label, [confidence[i]]\n                    else:\n                        active_scores.append(confidence[i])\n                        if kind == 2:  # L\n                            flush(i)\n                            active_start, active_label, active_scores = None, None, []\n                if active_start is not None:\n                    flush(len(predictions) - 1)\n            cursor += len(group)\n    if was_training:\n        model.train()\n    ranked = sorted(candidates.items(), key=lambda item: item[1], reverse=True)\n    if config.get(\'return_candidate_scores\', False):\n        return [[a, b, name, float(score)] for (a, b, name), score in ranked]\n    accepted = []\n    by_label = config.get(\'confidence_threshold_by_label\') or {}\n    for (a, b, name), score in ranked:\n        if score < float(by_label.get(name, config.get(\'confidence_threshold\', .45))):\n            continue\n        if any(a < other_b and other_a < b for other_a, other_b, _ in accepted):\n            continue\n        accepted.append((a, b, name))\n    return [list(item) for item in sorted(accepted)]\n'
IMPROVED_METHOD_SOURCES = {1: '"""Phase-2 experiment 01: full_finetune on Phase-1 base 01.\n\nAI-assisted reference code; disclose its use under course rules.\n"""\nfrom phase2_method_common import EntityTagger, collate, predict\n\n\nMETHOD = \'full_finetune\'\nBASE_ID = \'01\'\n\n\ndef build_task_model(base_model, tokenizer, labels, config, *, teacher_model=None,\n                     teacher_tokenizer=None):\n    return EntityTagger(base_model, BASE_ID, labels, METHOD, config, teacher=None)\n\n\ndef collate_batch(rows, tokenizer, labels, config, *, teacher_tokenizer=None):\n    return collate(rows, tokenizer, labels, config,\n                   teacher_tokenizer=teacher_tokenizer if METHOD == \'knowledge_distillation\' else None)\n\n\ndef predict_spans(model, text, tokenizer, labels, config):\n    return predict(model, text, tokenizer, labels, config)\n', 3: '"""Phase-2 experiment 03: lora on Phase-1 base 03.\n\nAI-assisted reference code; disclose its use under course rules.\n"""\nfrom phase2_method_common import EntityTagger, collate, predict\n\n\nMETHOD = \'lora\'\nBASE_ID = \'03\'\n\n\ndef build_task_model(base_model, tokenizer, labels, config, *, teacher_model=None,\n                     teacher_tokenizer=None):\n    return EntityTagger(base_model, BASE_ID, labels, METHOD, config, teacher=None)\n\n\ndef collate_batch(rows, tokenizer, labels, config, *, teacher_tokenizer=None):\n    return collate(rows, tokenizer, labels, config,\n                   teacher_tokenizer=teacher_tokenizer if METHOD == \'knowledge_distillation\' else None)\n\n\ndef predict_spans(model, text, tokenizer, labels, config):\n    return predict(model, text, tokenizer, labels, config)\n', 5: '"""Phase-2 experiment 05: adapter on Phase-1 base 01.\n\nAI-assisted reference code; disclose its use under course rules.\n"""\nfrom phase2_method_common import EntityTagger, collate, predict\n\n\nMETHOD = \'adapter\'\nBASE_ID = \'01\'\n\n\ndef build_task_model(base_model, tokenizer, labels, config, *, teacher_model=None,\n                     teacher_tokenizer=None):\n    return EntityTagger(base_model, BASE_ID, labels, METHOD, config, teacher=None)\n\n\ndef collate_batch(rows, tokenizer, labels, config, *, teacher_tokenizer=None):\n    return collate(rows, tokenizer, labels, config,\n                   teacher_tokenizer=teacher_tokenizer if METHOD == \'knowledge_distillation\' else None)\n\n\ndef predict_spans(model, text, tokenizer, labels, config):\n    return predict(model, text, tokenizer, labels, config)\n'}

"""Embedded Run-All orchestration for three improved Phase-2 NER submissions.

This cell is appended after the verified Phase-2 archive, data, model, training,
and packaging definitions.  It never mutates the previous Phase-2 workspace.
"""
from contextlib import suppress


IMPROVED_ROOT = Path(WORK_ROOT)
IMPROVED_SLOTS = (1, 3, 5)
CV_FOLDS = 3
CURRENT_STAGE_KEY = 'not_started'
IMPROVEMENT_VERSION = 'phase2-improved-v1'
SAFETY_DEADLINE_SECONDS = 31_200  # leave time before a ten-hour session limit
TRAINING_RESERVE_SECONDS = 2_700  # preserve time for test inference/packaging


def _improved_recovery_name(slot: int) -> str:
    return f'phase2_improved_{CURRENT_STAGE_KEY}_submission{slot:02d}_recovery.zip'


recovery_name = _improved_recovery_name


def _improved_common_module_path() -> Path:
    return IMPROVED_ROOT / 'method_source' / 'phase2_method_common.py'


common_module_path = _improved_common_module_path


def _prepare_improved_sources() -> None:
    common_path = _improved_common_module_path()
    common_path.parent.mkdir(parents=True, exist_ok=True)
    content = IMPROVED_COMMON_SOURCE.encode('utf-8')
    if common_path.exists():
        if common_path.read_bytes() != content:
            raise RuntimeError('Improved shared method source changed after initialization')
    else:
        with tempfile.NamedTemporaryFile(dir=common_path.parent, delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(content)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, common_path)
    for slot in IMPROVED_SLOTS:
        filename = f'phase2_method_{slot:02d}.py'
        source_path = IMPROVED_ROOT / 'method_source' / filename
        expected = IMPROVED_METHOD_SOURCES[slot].encode('utf-8')
        if source_path.exists():
            if source_path.read_bytes() != expected:
                raise RuntimeError(f'{filename} changed after improvement initialization')
        else:
            with tempfile.NamedTemporaryFile(dir=source_path.parent, delete=False) as handle:
                temporary = Path(handle.name)
                handle.write(expected)
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temporary, source_path)
        METHOD_SOURCE_OVERRIDES[slot] = str(source_path)
    print('Improvement method sources:', METHOD_SOURCE_OVERRIDES)


def _folds(data: Mapping[str, Any]) -> list[list[str]]:
    source1 = set(load_json(data['paths']['fold1-text.json']))
    source2 = set(load_json(data['paths']['fold2-text.json']))
    if source1 & source2 or source1 | source2 != set(data['train_text']):
        raise RuntimeError('Fold-source ID audit failed')
    groups = [sorted(source1), sorted(source2)]
    folds = [[] for _ in range(CV_FOLDS)]
    for group_index, group in enumerate(groups):
        random.Random(SEED + 100 + group_index).shuffle(group)
        for position, identifier in enumerate(group):
            folds[position % CV_FOLDS].append(identifier)
    for fold in folds:
        fold.sort()
    if sorted(identifier for fold in folds for identifier in fold) != sorted(data['train_text']):
        raise RuntimeError('Validation folds are incomplete or overlapping')
    manifest = {'version': IMPROVEMENT_VERSION, 'dataset_fingerprint': data['fingerprint'],
                'folds': folds, 'seed': SEED, 'source_stratified': True,
                'official_test_excluded': True}
    path = IMPROVED_ROOT / 'cv_folds.json'
    if path.exists() and load_json(path) != manifest:
        raise RuntimeError('Existing improvement folds differ; refusing unsafe resume')
    atomic_json(path, manifest)
    return folds


def _window_bounds(length: int, size: int, stride: int) -> list[tuple[int, int]]:
    if length <= size:
        return [(0, length)]
    starts = list(range(0, length - size + 1, stride))
    if starts[-1] != length - size:
        starts.append(length - size)
    return [(start, min(start + size, length)) for start in starts]


def _windowed_training_data(data, train_ids, tokenizer, config, stage_key):
    """Create stable, exhaustive document windows; validation remains untouched."""
    train_text = dict(data['train_text'])
    train_gold = dict(data['train_gold'])
    window_ids = []
    rare = {'APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT'}
    rare_ids = []
    uncovered = []
    size, stride = int(config['max_tokens']), int(config['window_stride'])
    # Leave a few BPE positions for retokenization at substring boundaries.
    core_size = size - 8
    for identifier in sorted(train_ids):
        text = data['train_text'][identifier]
        encoding = tokenizer.backend.encode(text, add_special_tokens=False)
        offsets = [(int(a), int(b)) for a, b in encoding.offsets]
        if not offsets:
            continue
        bounds = _window_bounds(len(offsets), core_size, min(stride, core_size))
        gold = data['train_gold'][identifier]
        covered = [False] * len(gold)
        for window_index, (lo, hi) in enumerate(bounds):
            char_lo, char_hi = offsets[lo][0], offsets[hi - 1][1]
            if char_hi <= char_lo:
                continue
            spans = []
            for gold_index, (a, b, label) in enumerate(gold):
                if char_lo <= a and b <= char_hi:
                    spans.append((a - char_lo, b - char_lo, label))
                    covered[gold_index] = True
            window_id = f'{identifier}__{window_index:04d}'
            train_text[window_id] = text[char_lo:char_hi]
            train_gold[window_id] = spans
            window_ids.append(window_id)
            if any(label in rare for _, _, label in spans):
                rare_ids.append(window_id)
        # Add an entity-centred window where a boundary fell through the
        # regular grid.  It keeps the entity complete without using test data.
        for gold_index, okay in enumerate(covered):
            if okay:
                continue
            a, b, label = gold[gold_index]
            touching = [i for i, (x, y) in enumerate(offsets) if y > a and x < b]
            if not touching or touching[-1] - touching[0] + 1 > core_size:
                uncovered.append((identifier, gold[gold_index]))
                continue
            lo = max(0, min(touching[0], len(offsets) - core_size))
            hi = min(len(offsets), lo + core_size)
            char_lo, char_hi = offsets[lo][0], offsets[hi - 1][1]
            if not char_lo <= a or not b <= char_hi:
                uncovered.append((identifier, gold[gold_index]))
                continue
            window_id = f'{identifier}__entity{gold_index:04d}'
            train_text[window_id] = text[char_lo:char_hi]
            train_gold[window_id] = [(x - char_lo, y - char_lo, name)
                                     for x, y, name in gold if char_lo <= x and y <= char_hi]
            window_ids.append(window_id)
            if label in rare:
                rare_ids.append(window_id)
    if uncovered:
        # An entity longer than a model window cannot be learned through this
        # token-tagging setup.  Surface it rather than silently claiming coverage.
        atomic_json(IMPROVED_ROOT / f'{stage_key}_uncovered_spans.json', uncovered)
        print(f'{stage_key}: {len(uncovered)} gold spans not wholly contained in a window')
    # Cap the extra rare-label visits so that ordinary text remains represented.
    extra_count = min(len(rare_ids), max(1, len(window_ids) // 6))
    extra = random.Random(SEED + len(window_ids)).sample(rare_ids, extra_count) if extra_count else []
    for index, source_id in enumerate(extra):
        duplicate_id = f'{source_id}__rare{index:05d}'
        train_text[duplicate_id] = train_text[source_id]
        train_gold[duplicate_id] = train_gold[source_id]
        window_ids.append(duplicate_id)
    if not window_ids:
        raise RuntimeError('No legal-NER training windows were built')
    fingerprint = hashlib.sha256(stable_json({
        'version': IMPROVEMENT_VERSION, 'original': data['fingerprint'],
        'stage': stage_key, 'train_ids': sorted(train_ids), 'window_ids': window_ids,
        'size': size, 'stride': stride,
    }).encode()).hexdigest()
    staged = dict(data)
    staged['train_text'] = train_text
    staged['train_gold'] = train_gold
    staged['fingerprint'] = fingerprint
    return staged, window_ids


def _config_for(slot: int, fold_index: int | None, num_windows: int,
                final_epochs: int | None = None) -> dict[str, Any]:
    config = dict(TRAINING_CONFIGS[slot])
    config.update(max_tokens=384, window_stride=288, batch_size=16,
                  gradient_accumulation_steps=1,
                  outside_class_weight=0.6, inference_window_batch_size=8,
                  confidence_threshold=0.45,
                  seed=SEED + slot * 100 + (fold_index if fold_index is not None else 10))
    if slot == 5:
        config['learning_rate'] = 1e-4
    epochs = final_epochs if final_epochs is not None else {1: 16, 3: 20, 5: 10}[slot]
    steps_per_epoch = math.ceil(num_windows / config['batch_size'])
    config.update(max_steps=max(1, epochs * steps_per_epoch),
                  validation_interval=steps_per_epoch,
                  checkpoint_interval=max(25, steps_per_epoch),
                  recovery_interval=max(50, 2 * steps_per_epoch),
                  improvement_version=IMPROVEMENT_VERSION,
                  exhaustive_window_epochs=epochs,
                  training_window_count=num_windows)
    return config


def _stage_root(stage_key: str) -> Path:
    return IMPROVED_ROOT / 'stages' / stage_key


def _activate_stage(stage_key: str) -> None:
    global WORK_ROOT, CURRENT_STAGE_KEY
    CURRENT_STAGE_KEY = stage_key
    WORK_ROOT = _stage_root(stage_key)
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    RESTORED_RECOVERY_SLOTS.clear()


def _run_stage(spec, data, train_ids, val_ids, stage_key, fold_index,
               deadline, final_epochs=None):
    slot = spec['slot']
    _activate_stage(stage_key)
    base, tokenizer, _meta = load_phase1_base(spec['base_id'], 'cpu')
    del base
    provisional = _config_for(slot, fold_index, 1, final_epochs)
    staged, windows = _windowed_training_data(data, train_ids, tokenizer, provisional, stage_key)
    config = _config_for(slot, fold_index, len(windows), final_epochs)
    TRAINING_CONFIGS[slot] = config
    if fold_index is not None:
        sealed_path = WORK_ROOT / 'cv_sealed.json'
        if sealed_path.exists():
            sealed = load_json(sealed_path)
            selected_path = experiment_directory(slot) / 'selected_model.pt'
            if (sealed.get('stage') != stage_key or sealed.get('config') != config or
                    not selected_path.is_file() or
                    sealed.get('selected_sha256') != sha256_file(selected_path)):
                raise RuntimeError(f'{stage_key}: sealed CV checkpoint failed integrity check')
            return staged, config, {'status': 'complete', 'step': sealed['selected_step'],
                                    'sealed_cv': True}
    print(f'{stage_key}: {len(windows)} training windows, '
          f'{config["max_steps"]} maximum steps, {len(val_ids)} validation documents')
    result = train_one_experiment(spec, staged, windows, val_ids, deadline)
    if result['status'] != 'complete':
        return staged, config, result
    selected_path = experiment_directory(slot) / 'selected_model.pt'
    if not selected_path.is_file():
        raise RuntimeError(f'{stage_key}: selected checkpoint missing after training')
    return staged, config, result


def _collect_cv_candidates(spec, staged, config, val_ids, stage_key):
    path = IMPROVED_ROOT / 'cv_candidates' / f'{stage_key}.json'
    directory = experiment_directory(spec['slot'])
    method, method_sha = load_student_method(spec, directory / spec['module_filename'])
    selected_path = directory / 'selected_model.pt'
    selected_sha = sha256_file(selected_path)
    expected = {'stage': stage_key, 'selected_sha256': selected_sha,
                'ids': sorted(val_ids), 'version': IMPROVEMENT_VERSION}
    if path.exists():
        record = load_json(path)
        if record.get('provenance') != expected:
            raise RuntimeError(f'{stage_key}: stale candidate cache')
        candidates = record['candidates']
        if set(candidates) == set(val_ids) and not record.get('incomplete'):
            return candidates
    else:
        candidates = {}
    model, tokenizer, _, _ = load_selected_task(
        spec, staged, config, method, method_sha, 'cuda:0')
    candidate_config = dict(config, return_candidate_scores=True)
    try:
        for identifier in sorted(val_ids):
            if identifier in candidates:
                continue
            candidates[identifier] = method.predict_spans(
                model, staged['train_text'][identifier], tokenizer,
                staged['labels'], candidate_config)
            if len(candidates) % 5 == 0:
                atomic_json(path, {'provenance': expected, 'candidates': candidates,
                                   'incomplete': True})
    finally:
        del model
        torch.cuda.empty_cache()
    atomic_json(path, {'provenance': expected, 'candidates': candidates})
    return candidates


def _seal_cv_stage(spec, config, stage_key):
    """Keep the verified recovery ZIP, selected model, and history; free live optimizer state."""
    directory = experiment_directory(spec['slot'])
    selected_path = directory / 'selected_model.pt'
    candidate_path = IMPROVED_ROOT / 'cv_candidates' / f'{stage_key}.json'
    recovery_path = directory / recovery_name(spec['slot'])
    candidate_record = load_json(candidate_path)
    if candidate_record.get('incomplete') or not recovery_path.is_file():
        raise RuntimeError(f'{stage_key}: refusing to seal incomplete CV artifacts')
    selected = torch.load(selected_path, map_location='cpu', weights_only=True)
    seal = {'stage': stage_key, 'config': config,
            'selected_sha256': sha256_file(selected_path),
            'selected_step': int(selected['step']),
            'candidate_sha256': sha256_file(candidate_path),
            'recovery_sha256': sha256_file(recovery_path)}
    seal_path = WORK_ROOT / 'cv_sealed.json'
    if seal_path.exists() and load_json(seal_path) != seal:
        raise RuntimeError(f'{stage_key}: existing CV seal differs')
    atomic_json(seal_path, seal)
    for filename in ('checkpoint.latest.pt', 'checkpoint.latest.previous_known_good.pt'):
        target = directory / filename
        if target.is_file():
            target.unlink()


def _decode_candidates(candidate_map, thresholds, default=0.45):
    predictions = {}
    for identifier, rows in candidate_map.items():
        accepted = []
        for a, b, label, score in sorted(rows, key=lambda row: -row[3]):
            if score < thresholds.get(label, default):
                continue
            if any(a < other_b and other_a < b for other_a, other_b, _ in accepted):
                continue
            accepted.append((int(a), int(b), label))
        predictions[identifier] = [list(item) for item in sorted(accepted)]
    return predictions


def _calibrate(slot, data, fold_candidates, folds):
    labels = data['labels']
    all_candidates = {identifier: rows for group in fold_candidates for identifier, rows in group.items()}
    if set(all_candidates) != set(data['train_text']):
        raise RuntimeError(f'Slot {slot}: out-of-fold candidate IDs are incomplete')
    gold = data['train_gold']
    grid = (0.30, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80)
    thresholds = {label: 0.45 for label in labels}

    def score(candidate_subset, thresholds_to_use):
        predictions = _decode_candidates(candidate_subset, thresholds_to_use)
        expected = {identifier: gold[identifier] for identifier in candidate_subset}
        return strict_macro_prf(predictions, expected, labels)['strict_macro_f1_local']

    baseline = score(all_candidates, thresholds)
    for label in labels:
        best_threshold, best_score = thresholds[label], score(all_candidates, thresholds)
        for trial in grid:
            proposed = dict(thresholds, **{label: trial})
            measured = score(all_candidates, proposed)
            if measured > best_score + 1e-9:
                best_threshold, best_score = trial, measured
        thresholds[label] = best_threshold
    tuned = score(all_candidates, thresholds)
    fold_scores = []
    for candidate_subset in fold_candidates:
        fold_scores.append({'baseline': score(candidate_subset, {label: 0.45 for label in labels}),
                            'tuned': score(candidate_subset, thresholds)})
    # Avoid a fragile calibration that only helps the pooled score but hurts
    # two of the three held-out document groups.
    hurts = sum(item['tuned'] + 1e-9 < item['baseline'] for item in fold_scores)
    if tuned <= baseline or hurts >= 2:
        thresholds = {label: 0.45 for label in labels}
        tuned = baseline
    report = {'slot': slot, 'candidate_documents': len(all_candidates),
              'baseline_oof_macro_f1': baseline, 'calibrated_oof_macro_f1': tuned,
              'fold_scores': fold_scores, 'confidence_threshold_by_label': thresholds,
              'note': 'Thresholds optimized on OOF predictions; reported tuned score is selection-biased.'}
    atomic_json(IMPROVED_ROOT / f'submission_{slot:02d}_cv_calibration.json', report)
    return report


def _final_epoch_choice(slot, fold_steps, fold_window_counts) -> int:
    epochs = [step / max(1, math.ceil(count / 16))
              for step, count in zip(fold_steps, fold_window_counts)]
    choice = int(round(float(np.median(epochs))))
    return max(3, min(choice, {1: 16, 3: 20, 5: 10}[slot]))


def _promote_final_checkpoint(spec, config, cv_report):
    """Use the CV-chosen training duration, not in-sample final validation."""
    directory = experiment_directory(spec['slot'])
    latest = torch.load(directory / 'checkpoint.latest.pt', map_location='cpu', weights_only=True)
    selected_path = directory / 'selected_model.pt'
    selected = torch.load(selected_path, map_location='cpu', weights_only=True)
    if latest['step'] != config['max_steps'] or selected['training_config'] != config:
        raise RuntimeError('Final checkpoint/config is incomplete or inconsistent')
    result_path = directory / 'training_result.json'
    result = load_json(result_path)
    result['best_strict_macro_f1_local'] = None
    result['selection_note'] = 'Final fit used all public labeled documents; CV selected the duration.'
    result['cv_baseline_oof_macro_f1'] = cv_report['baseline_oof_macro_f1']
    result['cv_calibrated_oof_macro_f1'] = cv_report['calibrated_oof_macro_f1']
    atomic_json(result_path, result)
    if selected.get('selection_note') == 'Full-data final weights at CV-selected epoch; no honest final holdout score':
        if selected['step'] != latest['step']:
            raise RuntimeError('Previously promoted selected checkpoint has wrong step')
        return
    selected['model_state'] = latest['model_state']
    selected['step'] = latest['step']
    selected['strict_macro_f1_local'] = None
    selected['selection_note'] = 'Full-data final weights at CV-selected epoch; no honest final holdout score'
    selected['cv_baseline_oof_macro_f1'] = cv_report['baseline_oof_macro_f1']
    selected['cv_calibrated_oof_macro_f1'] = cv_report['calibrated_oof_macro_f1']
    atomic_torch_save(selected_path, selected)
    export_recovery(spec['slot'])


def _predict_final(spec, staged, train_config, cv_report, deadline):
    slot = spec['slot']
    directory = experiment_directory(slot)
    method, method_sha = load_student_method(spec, directory / spec['module_filename'])
    selected_path = directory / 'selected_model.pt'
    decoding_config = dict(train_config)
    decoding_config['confidence_threshold_by_label'] = cv_report['confidence_threshold_by_label']
    provenance = {'slot': slot, 'selected_sha256': sha256_file(selected_path),
                  'method_sha256': method_sha,
                  'test_file_sha256': sha256_file(staged['paths']['fold3-text.json']),
                  'decoding_thresholds': decoding_config['confidence_threshold_by_label']}
    partial_path = directory / 'predictions.improved.partial.json'
    meta_path = directory / 'predictions.improved.provenance.json'
    if meta_path.exists() and load_json(meta_path) != provenance:
        raise RuntimeError('Partial inference provenance changed; refusing stale predictions')
    if partial_path.exists():
        if not meta_path.exists():
            raise RuntimeError('Partial predictions lack provenance')
        predictions = load_json(partial_path)
    else:
        predictions = {}
        atomic_json(meta_path, provenance)
    if set(predictions) - set(staged['test_text']):
        raise RuntimeError('Partial predictions contain unexpected official IDs')
    for identifier, spans in predictions.items():
        validate_prediction_map({identifier: spans},
                                {identifier: staged['test_text'][identifier]}, staged['labels'])
    if len(predictions) != len(staged['test_text']):
        model, tokenizer, _, _ = load_selected_task(
            spec, staged, train_config, method, method_sha, 'cuda:0')
        try:
            failures = {}
            for attempt in range(3):
                pending = [identifier for identifier in staged['test_text'] if identifier not in predictions]
                if not pending:
                    break
                for identifier in pending:
                    if time.monotonic() >= deadline:
                        export_recovery(slot)
                        return None
                    try:
                        text = staged['test_text'][identifier]
                        spans = [] if not text else method.predict_spans(
                            model, text, tokenizer, staged['labels'], decoding_config)
                        validate_prediction_map({identifier: spans},
                                                {identifier: text}, staged['labels'])
                        predictions[identifier] = spans
                        atomic_json(partial_path, predictions)
                        failures.pop(identifier, None)
                        append_jsonl(directory / 'inference_attempts.jsonl',
                                     {'id': identifier, 'attempt': attempt + 1, 'status': 'success'})
                    except Exception as exc:
                        failures[identifier] = f'{type(exc).__name__}: {exc}'
                        append_jsonl(directory / 'inference_attempts.jsonl',
                                     {'id': identifier, 'attempt': attempt + 1,
                                      'status': 'failure', 'error': failures[identifier]})
                        torch.cuda.empty_cache()
                atomic_json(directory / 'inference_failures.json', failures)
            if len(predictions) != len(staged['test_text']):
                export_recovery(slot)
                raise RuntimeError(f'Slot {slot}: {len(staged["test_text"])-len(predictions)} '
                                   'official documents remain unpredicted; partial state is saved')
        finally:
            del model
            torch.cuda.empty_cache()
    package = make_submission_zip(slot, predictions, staged, provenance)
    top_level = IMPROVED_ROOT / f'submission_{slot:02d}_improved.zip'
    if top_level.exists():
        with zipfile.ZipFile(top_level) as existing:
            if existing.namelist() != ['prediction.json'] or existing.testzip() is not None:
                raise RuntimeError(f'{top_level} is not a valid previous submission')
            old_prediction_sha = hashlib.sha256(existing.read('prediction.json')).hexdigest()
        if old_prediction_sha != package['prediction_sha256']:
            raise RuntimeError(f'{top_level} already exists with different predictions')
    if not top_level.exists():
        with tempfile.NamedTemporaryFile(dir=IMPROVED_ROOT, suffix='.zip', delete=False) as handle:
            temporary = Path(handle.name)
        try:
            shutil.copyfile(package['zip_path'], temporary)
            os.replace(temporary, top_level)
        finally:
            temporary.unlink(missing_ok=True)
    upload_copy = IMPROVED_ROOT / 'final_submissions' / f'submission_{slot:02d}' / 'submission.zip'
    upload_copy.parent.mkdir(parents=True, exist_ok=True)
    if upload_copy.exists() and sha256_file(upload_copy) != sha256_file(top_level):
        raise RuntimeError(f'{upload_copy} already exists with different bytes')
    if not upload_copy.exists():
        shutil.copyfile(top_level, upload_copy)
    package['download_zip'] = str(top_level)
    package['upload_zip'] = str(upload_copy)
    export_recovery(slot)
    return package


def run_improvement_all():
    global WORK_ROOT
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError('This Run-All plan requires two visible Kaggle GPUs')
    IMPROVED_ROOT.mkdir(parents=True, exist_ok=True)
    _prepare_improved_sources()
    data = load_official_phase2()
    folds = _folds(data)
    spec_by_slot = {spec['slot']: spec for spec in EXPERIMENTS}
    started = time.monotonic()
    total_deadline = started + SAFETY_DEADLINE_SECONDS
    training_deadline = total_deadline - TRAINING_RESERVE_SECONDS
    status_path = IMPROVED_ROOT / 'improvement_status.json'
    results = {}
    for slot in IMPROVED_SLOTS:
        spec = spec_by_slot[slot]
        fold_candidates, selected_steps, fold_window_counts = [], [], []
        for fold_index, val_ids in enumerate(folds):
            if time.monotonic() >= training_deadline:
                atomic_json(status_path, {'status': 'TIME_GUARD', 'next': f'cv_{slot:02d}_{fold_index}',
                                          'completed': results})
                return results
            train_ids = sorted(set(data['train_text']) - set(val_ids))
            stage_key = f'cv{fold_index}_slot{slot:02d}'
            staged, config, result = _run_stage(
                spec, data, train_ids, val_ids, stage_key, fold_index, training_deadline)
            if result['status'] != 'complete':
                atomic_json(status_path, {'status': 'TIME_GUARD', 'next': stage_key,
                                          'completed': results})
                return results
            selected = torch.load(experiment_directory(slot) / 'selected_model.pt',
                                  map_location='cpu', weights_only=True)
            selected_steps.append(int(selected['step']))
            fold_window_counts.append(int(config['training_window_count']))
            fold_candidates.append(_collect_cv_candidates(spec, staged, config, val_ids, stage_key))
            _seal_cv_stage(spec, config, stage_key)
            atomic_json(status_path, {'status': 'CV_IN_PROGRESS', 'last_completed': stage_key,
                                      'completed': results})
        cv_report = _calibrate(slot, data, fold_candidates, folds)
        final_epochs = _final_epoch_choice(slot, selected_steps, fold_window_counts)
        stage_key = f'final_slot{slot:02d}'
        if time.monotonic() >= training_deadline:
            atomic_json(status_path, {'status': 'TIME_GUARD', 'next': stage_key,
                                      'completed': results})
            return results
        # One training document is used only for the runner's smoke/periodic
        # diagnostic; every labeled document is included in final training.
        all_ids = sorted(data['train_text'])
        staged, config, result = _run_stage(
            spec, data, all_ids, all_ids[:1], stage_key, None,
            training_deadline, final_epochs=final_epochs)
        if result['status'] != 'complete':
            atomic_json(status_path, {'status': 'TIME_GUARD', 'next': stage_key,
                                      'completed': results})
            return results
        _promote_final_checkpoint(spec, config, cv_report)
        package = _predict_final(spec, staged, config, cv_report, total_deadline)
        if package is None:
            atomic_json(status_path, {'status': 'TIME_GUARD', 'next': f'inference_{slot:02d}',
                                      'completed': results})
            return results
        results[str(slot)] = {'cv': cv_report, 'final_epochs': final_epochs, 'package': package}
        atomic_json(status_path, {'status': 'IN_PROGRESS', 'completed': results})
        print(f'Submission {slot:02d} improved package: {package["download_zip"]}')
    if set(results) != {'1', '3', '5'}:
        raise RuntimeError('Improved completion manifest cannot omit a planned slot')
    hashes = [results[str(slot)]['package']['prediction_sha256'] for slot in IMPROVED_SLOTS]
    if len(set(hashes)) != 3:
        raise RuntimeError('Improved submissions have duplicate prediction contents')
    atomic_json(status_path, {'status': 'READY_FOR_MANUAL_CODABENCH_UPLOAD',
                              'completed': results,
                              'note': 'Official score cannot be known before Codabench evaluation.'})
    return results


try:
    IMPROVEMENT_RESULTS = run_improvement_all()
except Exception as improvement_error:
    atomic_json(IMPROVED_ROOT / 'improvement_status.json',
                {'status': 'ERROR', 'error': f'{type(improvement_error).__name__}: {improvement_error}',
                 'active_stage': CURRENT_STAGE_KEY,
                 'note': 'Earlier checkpoints and recovery ZIPs are preserved; inspect traceback.'})
    raise
print('Phase-2 improvement status:', load_json(IMPROVED_ROOT / 'improvement_status.json')['status'])
